#### Stage-1 importer (ONLY drug table)
drug ✅ (root table; everything else hangs from this)

In [13]:
# Run this single cell to install all dependencies while suppressing the root user warning
!pip install -q --root-user-action=ignore numpy pandas scipy scikit-learn networkx matplotlib seaborn tqdm psutil lxml

In [14]:
# 1. Reset OS directory permissions to allow writing temporary files
!if [ -d /tmp ]; then chmod 1777 /tmp; fi

# 2. Synchronize your Linux package tool safely 
!apt-get update -y

# 3. Install the required MySQL/MariaDB operating system library dependencies
!apt-get install -y libmysqlclient-dev pkg-config build-essential

# 4. Install the pipeline suite alongside the specific missing mysql-connector driver
!pip install --root-user-action=ignore numpy pandas scipy scikit-learn networkx matplotlib seaborn tqdm psutil mysqlclient mysql-connector-python

Hit:1 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:2 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists... Done
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libmysqlclient-dev is already the newest version (8.0.46-0ubuntu0.24.04.3).
pkg-config is already the newest version (1.8.1-2build1).
build-essential is already the newest version (12.10ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 139 not upgraded.


In [15]:
# 1. Install ipywidgets to provide the missing IProgress backend
!pip install --root-user-action=ignore ipywidgets

# 2. Refresh the Jupyter widget extensions for standard notebooks
!jupyter nbextension enable --py widgetsnbextension

usage: jupyter [-h] [--version] [--config-dir] [--data-dir] [--runtime-dir]
               [--paths] [--json] [--debug]
               [subcommand]

Jupyter: Interactive Computing

positional arguments:
  subcommand     the subcommand to launch

options:
  -h, --help     show this help message and exit
  --version      show the versions of core jupyter packages and exit
  --config-dir   show Jupyter config dir
  --data-dir     show Jupyter data dir
  --runtime-dir  show Jupyter runtime dir
  --paths        show all Jupyter paths. Add --json for machine-readable
                 format.
  --json         output paths as machine-readable json
  --debug        output debug information about paths

Available subcommands: dejavu events execute kernel kernelspec lab
labextension labhub migrate nbconvert notebook run server troubleshoot trust

Jupyter command `jupyter-nbextension` not found.


In [16]:
import sys
# Force the runtime path environment to refresh and detect the widget libraries
!{sys.executable} -m pip check

No broken requirements found.


In [26]:
# =========================================================================
# Stage 1: Load drug table (FULL, reliable) - Consolidated Execution Node
# =========================================================================
import time
import sys
from lxml import etree as ET
import mysql.connector

# ─── ROBUST PROGRESS BAR CLASS ENFORCEMENT ────────────────────────────────
# Dynamically binds the correct callable class interface to prevent 
# module-level collision exceptions during notebook execution runtime.
from tqdm import tqdm as tqdm_base
try:
    from tqdm.notebook import tqdm as tqdm_notebook_class
    # Validate instance execution capabilities
    p_test = tqdm_notebook_class(total=1, disable=True)
    p_test.close()
    tqdm = tqdm_notebook_class
except Exception:
    tqdm = tqdm_base
# ──────────────────────────────────────────────────────────────────────────

# --------- EDIT THESE ----------
XML_PATH = r"db/drugbank.xml"

MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "niraj"
MYSQL_PASSWORD = "Niraj@123"
MYSQL_DB = "drugbank"

COMMIT_EVERY = 1000
SHOW_EVERY = 10000

# Set True for clean reload (deletes all rows from drug first)
WIPE_DRUG_TABLE = False
# -------------------------------

def get_conn():
    return mysql.connector.connect(
        host=MYSQL_HOST, port=MYSQL_PORT, user=MYSQL_USER,
        password=MYSQL_PASSWORD, database=MYSQL_DB, autocommit=False
    )

def strip_ns(tag: str) -> str:
    return tag.split("}", 1)[1] if "}" in tag else tag

def detect_namespace(xml_path: str):
    ctx = ET.iterparse(xml_path, events=("start",))
    _, root = next(ctx)
    if root.tag.startswith("{"):
        return root.tag.split("}")[0].strip("{")
    return None

def nstag_factory(ns):
    def nstag(local: str) -> str:
        return f"{{{ns}}}{local}" if ns else local
    return nstag

def text(elem):
    if elem is None or elem.text is None:
        return None
    t = elem.text.strip()
    return t if t else None

def to_float(s):
    if s is None:
        return None
    try:
        return float(s)
    except:
        return None

def bool_from_text(s):
    if s is None:
        return None
    v = str(s).strip().lower()
    if v in ("true", "1", "yes"): return True
    if v in ("false", "0", "no"): return False
    return None

def cleanup_lxml(elem):
    elem.clear()
    while elem.getprevious() is not None:
        del elem.getparent()[0]

def count_total_drugs(xml_path: str) -> int:
    total = 0
    p = tqdm(total=None, unit="drug", dynamic_ncols=True, desc="Pre-pass: counting <drug>")
    try:
        ctx = ET.iterparse(xml_path, events=("end",), huge_tree=True)
        for _, e in ctx:
            if strip_ns(e.tag) == "drug":
                total += 1
                p.update(1)
                cleanup_lxml(e)
    finally:
        try: p.close()
        except: pass
    return total

def ensure_unique_index_on_drug(conn):
    """
    Ensures drug.primary_drugbank_id has a UNIQUE index.
    This is REQUIRED for ON DUPLICATE KEY UPDATE to work correctly.
    """
    cur = conn.cursor()
    cur.execute("SHOW INDEX FROM drug WHERE Column_name='primary_drugbank_id'")
    rows = cur.fetchall()
    has_unique = any(int(r[1]) == 0 for r in rows)  # Non_unique == 0
    if not has_unique:
        print("[INFO] Adding UNIQUE index on drug(primary_drugbank_id)...")
        cur.execute("ALTER TABLE drug ADD UNIQUE KEY uq_drug_primary_id (primary_drugbank_id)")
        conn.commit()
    cur.close()

DRUG_UPSERT_SQL = """
INSERT INTO drug (
  primary_drugbank_id, name, description, cas_number, unii,
  average_mass, monoisotopic_mass, state,
  synthesis_reference, indication, pharmacodynamics, mechanism_of_action,
  toxicity, metabolism, absorption, half_life, protein_binding,
  route_of_elimination, volume_of_distribution, clearance,
  fda_label, msds
)
VALUES (%s,%s,%s,%s,%s,
        %s,%s,%s,
        %s,%s,%s,%s,
        %s,%s,%s,%s,%s,
        %s,%s,%s,
        %s,%s)
ON DUPLICATE KEY UPDATE
  name = VALUES(name),
  description = VALUES(description),
  cas_number = VALUES(cas_number),
  unii = VALUES(unii),
  average_mass = VALUES(average_mass),
  monoisotopic_mass = VALUES(monoisotopic_mass),
  state = VALUES(state),
  synthesis_reference = VALUES(synthesis_reference),
  indication = VALUES(indication),
  pharmacodynamics = VALUES(pharmacodynamics),
  mechanism_of_action = VALUES(mechanism_of_action),
  toxicity = VALUES(toxicity),
  metabolism = VALUES(metabolism),
  absorption = VALUES(absorption),
  half_life = VALUES(half_life),
  protein_binding = VALUES(protein_binding),
  route_of_elimination = VALUES(route_of_elimination),
  volume_of_distribution = VALUES(volume_of_distribution),
  clearance = VALUES(clearance),
  fda_label = VALUES(fda_label),
  msds = VALUES(msds)
"""

def import_stage1_drug(xml_path: str):
    ns = detect_namespace(xml_path)
    nstag = nstag_factory(ns)
    print("Detected namespace:", ns)

    total = count_total_drugs(xml_path)
    print(f"Total <drug> in XML: {total:,}")

    conn = get_conn()
    cur = conn.cursor()

    # Ensure uniqueness so upsert behaves correctly
    ensure_unique_index_on_drug(conn)

    if WIPE_DRUG_TABLE:
        print("[CAUTION] Deleting all rows from drug...")
        cur.execute("DELETE FROM drug")
        conn.commit()

    processed = 0
    errors = 0
    inserted = 0
    updated = 0
    unchanged = 0
    t0 = time.time()
    pbar = tqdm(total=total, unit="drug", dynamic_ncols=True, desc="Stage 1: drug")

    try:
        ctx = ET.iterparse(xml_path, events=("end",), huge_tree=True)
        for _, elem in ctx:
            if strip_ns(elem.tag) != "drug":
                continue

            try:
                # ---- choose primary_drugbank_id deterministically ----
                ids = elem.findall(nstag("drugbank-id"))
                primary_id = None
                first_id = None
                for ide in ids:
                    did = text(ide)
                    if not did:
                        continue
                    if first_id is None:
                        first_id = did
                    if bool_from_text(ide.get("primary")):
                        primary_id = did
                        break
                primary_id = primary_id or first_id
                if not primary_id:
                    errors += 1
                    processed += 1
                    pbar.update(1)
                    cleanup_lxml(elem)
                    continue

                def f(tag):
                    return text(elem.find(nstag(tag)))

                row = (
                    primary_id,
                    f("name") or "",     # NOT NULL
                    f("description"),
                    f("cas-number"),
                    f("unii"),
                    to_float(f("average-mass")),
                    to_float(f("monoisotopic-mass")),
                    f("state"),
                    f("synthesis-reference"),
                    f("indication"),
                    f("pharmacodynamics"),
                    f("mechanism-of-action"),
                    f("toxicity"),
                    f("metabolism"),
                    f("absorption"),
                    f("half-life"),
                    f("protein-binding"),
                    f("route-of-elimination"),
                    f("volume-of-distribution"),
                    f("clearance"),
                    f("fda-label"),
                    f("msds"),
                )
                cur.execute(DRUG_UPSERT_SQL, row)

                # classify what happened
                cur.execute("SELECT ROW_COUNT()")
                rc = int(cur.fetchone()[0])
                if rc == 1:
                    inserted += 1
                elif rc == 2:
                    updated += 1
                else:
                    unchanged += 1

                processed += 1
                pbar.update(1)

                if processed % COMMIT_EVERY == 0:
                    conn.commit()
                if processed % SHOW_EVERY == 0:
                    dt = time.time() - t0
                    rate = processed / dt if dt > 0 else 0.0
                    print(f"[Stage1] {processed:,}/{total:,} rate={rate:.2f}/sec "
                          f"ins={inserted:,} upd={updated:,} same={unchanged:,} err={errors:,}")
            except Exception as e:
                errors += 1
                conn.rollback()
                print("ERROR on drug:", e)

            finally:
                cleanup_lxml(elem)

        conn.commit()

    finally:
        try: pbar.close()
        except: pass
        cur.close()
        conn.close()

    dt = time.time() - t0
    rate = processed / dt if dt > 0 else 0.0
    print("\nDONE Stage 1 (drug)")
    print(f"Processed:  {processed:,} (expected {total:,})")
    print(f"Inserted:   {inserted:,}")
    print(f"Updated:    {updated:,}")
    print(f"Unchanged:  {unchanged:,}")
    print(f"Errors:     {errors:,}")
    print(f"Elapsed:    {dt:.1f}s  Rate: {rate:.2f}/sec")

    # sanity check from DB
    conn2 = get_conn()
    cur2 = conn2.cursor()
    cur2.execute("SELECT COUNT(*), COUNT(DISTINCT primary_drugbank_id) FROM drug")
    total_rows, distinct_ids = cur2.fetchone()
    cur2.close(); conn2.close()
    print(f"DB check: drug rows={int(total_rows):,}, distinct primary_drugbank_id={int(distinct_ids):,}")

# RUN
import_stage1_drug(XML_PATH)

Detected namespace: http://www.drugbank.ca


Pre-pass: counting <drug>: 0drug [00:00, ?drug/s]

Total <drug> in XML: 73,687


Stage 1: drug:   0%|          | 0/73687 [00:00<?, ?drug/s]

[Stage1] 10,000/73,687 rate=1986.76/sec ins=342 upd=62 same=9,596 err=0
[Stage1] 20,000/73,687 rate=1345.42/sec ins=720 upd=207 same=19,073 err=0
[Stage1] 30,000/73,687 rate=856.26/sec ins=1,851 upd=617 same=27,532 err=0
[Stage1] 40,000/73,687 rate=973.97/sec ins=2,799 upd=745 same=36,456 err=0
[Stage1] 50,000/73,687 rate=1070.38/sec ins=3,759 upd=872 same=45,369 err=0
[Stage1] 60,000/73,687 rate=1115.32/sec ins=5,261 upd=966 same=53,773 err=0
[Stage1] 70,000/73,687 rate=824.05/sec ins=13,743 upd=1,039 same=55,218 err=0

DONE Stage 1 (drug)
Processed:  73,687 (expected 73,687)
Inserted:   17,430
Updated:    1,039
Unchanged:  55,218
Errors:     0
Elapsed:    88.2s  Rate: 835.51/sec
DB check: drug rows=17,430, distinct primary_drugbank_id=17,430


#### Stage 2 — Direct children of drug (FK → drug)
These only need drug_pk:
drugbank_id_map
drug_group
drug_classification (one row per drug)
drug_classification_alternative_parent
drug_classification_substituent
drug_synonym
drug_salt
product
packager
manufacturer
drug_category
drug_affected_organism
drug_dosage
drug_ahfs_code
drug_pdb_entry
drug_atc_code
drug_atc_level (depends on drug_atc_code via FK (drug_pk, atc_code))
drug_price
drug_patent
drug_food_interaction
drug_interaction
drug_external_identifier
drug_external_link
drug_property
reaction (depends on drug_pk)

Below is a Stage-2 only notebook-friendly solution that does exactly what you asked:
One pass per table (it re-reads / re-scans the XML for each table)
One function per Stage-2 table
Shows progress bar for each table pass
At the end, prints a final summary table:
how many rows were attempted / affected during that pass
the current row count in the DB for that table (SELECT COUNT(*))
✅ Assumption: Stage-1 (drug) is already loaded, so we can look up drug_pk from primary_drugbank_id.

## Cautious it will delete all the data from Stage - 2

In [23]:
import mysql.connector
from mysql.connector import Error

# =========================================================================
# CONFIGURATION - Must connect as ROOT to perform admin operations
# =========================================================================
ROOT_USER = "root"
ROOT_PASSWORD = ""  # Put your MySQL root password here (leave blank if none)
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306

NEW_USER = "niraj"
NEW_PASSWORD = "Niraj@123"
DATABASE_NAME = "drugbank"

def setup_database_and_user():
    conn = None
    cursor = None
    try:
        print("[INFO] Connecting to MySQL server as root...")
        conn = mysql.connector.connect(
            host=MYSQL_HOST,
            port=MYSQL_PORT,
            user=ROOT_USER,
            password=ROOT_PASSWORD,
            autocommit=True,
        )
        cursor = conn.cursor()
        print("[SUCCESS] Connected to MySQL server.")

        # 1. Create Database with UTF-8 support
        print(f"[INFO] Creating database '{DATABASE_NAME}'...")
        cursor.execute(f"""
            CREATE DATABASE IF NOT EXISTS `{DATABASE_NAME}` 
            CHARACTER SET utf8mb4 
            COLLATE utf8mb4_unicode_ci;
        """)
        print(f"[SUCCESS] Database '{DATABASE_NAME}' ready.")

        # 2. Ensure user 'niraj' exists for both localhost and '%'
        print(f"[INFO] Creating user '{NEW_USER}'...")
        cursor.execute(f"""
            CREATE USER IF NOT EXISTS '{NEW_USER}'@'localhost' IDENTIFIED BY '{NEW_PASSWORD}';
        """)
        cursor.execute(f"""
            CREATE USER IF NOT EXISTS '{NEW_USER}'@'%' IDENTIFIED BY '{NEW_PASSWORD}';
        """)
        print(f"[SUCCESS] User '{NEW_USER}' created/verified.")

        # 3. Grant ALL PRIVILEGES on drugbank database
        print(f"[INFO] Granting privileges on '{DATABASE_NAME}' to '{NEW_USER}'...")
        cursor.execute(f"""
            GRANT ALL PRIVILEGES ON `{DATABASE_NAME}`.* TO '{NEW_USER}'@'localhost';
        """)
        cursor.execute(f"""
            GRANT ALL PRIVILEGES ON `{DATABASE_NAME}`.* TO '{NEW_USER}'@'%';
        """)

        # 4. Apply privilege changes immediately
        cursor.execute("FLUSH PRIVILEGES;")
        print("[SUCCESS] Privileges granted and flushed successfully.")

        print("\n========================================================")
        print(f"Database '{DATABASE_NAME}' and User '{NEW_USER}' Setup Complete!")
        print("========================================================")

    except Error as db_err:
        print(f"\n[ERROR] Failed to execute administrative commands: {db_err}")
        print("Make sure ROOT_PASSWORD matches your MySQL root account password.")

    finally:
        if cursor:
            cursor.close()
        if conn:
            conn.close()

if __name__ == "__main__":
    setup_database_and_user()

[INFO] Connecting to MySQL server as root...

[ERROR] Failed to execute administrative commands: 1698 (28000): Access denied for user 'root'@'localhost'
Make sure ROOT_PASSWORD matches your MySQL root account password.


In [25]:
import mysql.connector

MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "niraj"
MYSQL_PASSWORD = "Niraj@123"
MYSQL_DB = "drugbank"

def get_conn():
    return mysql.connector.connect(
        host=MYSQL_HOST,
        port=MYSQL_PORT,
        user=MYSQL_USER,
        password=MYSQL_PASSWORD,
        database=MYSQL_DB,
        autocommit=True
    )

def wipe_entire_database():
    print(f"⚠️ WARNING: This will permanently delete ALL DATA from database '{MYSQL_DB}'.")
    confirm = input("Type WIPE ALL to confirm database clearing: ").strip()
    
    if confirm != "WIPE ALL":
        print("Operation cancelled. No tables were altered.")
        return

    conn = get_conn()
    cur = conn.cursor()

    try:
        # 1. Fetch all existing tables in the database dynamically
        cur.execute("""
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = %s
        """, (MYSQL_DB,))
        tables = [row[0] for row in cur.fetchall()]

        if not tables:
            print(f"\nDatabase '{MYSQL_DB}' is already completely empty (0 tables found).")
            return

        print(f"\n[INFO] Found {len(tables)} tables. Disabling foreign key checks...")
        
        # 2. Disable Foreign Key Checks globally for this session
        cur.execute("SET FOREIGN_KEY_CHECKS = 0;")

        # 3. Truncate each table
        cleared_tables = []
        failed_tables = []

        for table in tables:
            try:
                cur.execute(f"TRUNCATE TABLE `{table}`")
                cleared_tables.append(table)
                print(f"✓ Cleared: {table:40} -> TRUNCATE")
            except mysql.connector.Error:
                # Fallback to DELETE if TRUNCATE is restricted
                try:
                    cur.execute(f"DELETE FROM `{table}`")
                    cleared_tables.append(table)
                    print(f"✓ Cleared: {table:40} -> DELETE")
                except mysql.connector.Error as err:
                    failed_tables.append((table, str(err)))
                    print(f"✗ Failed:  {table:40} -> {err}")

        # 4. Re-enable Foreign Key Checks
        cur.execute("SET FOREIGN_KEY_CHECKS = 1;")

        print("\n" + "=" * 60)
        print("DATABASE WIPE SUMMARY")
        print(f"Total tables detected : {len(tables)}")
        print(f"Successfully emptied : {len(cleared_tables)}")
        print(f"Failed to empty      : {len(failed_tables)}")
        print("=" * 60)

        # 5. Sanity check row counts
        total_remaining_rows = 0
        for table in cleared_tables:
            cur.execute(f"SELECT COUNT(*) FROM `{table}`")
            count = cur.fetchone()[0]
            total_remaining_rows += count

        print(f"\nFinal Check: Total records remaining across all tables = {total_remaining_rows}")
        if total_remaining_rows == 0:
            print("🎉 SUCCESS: The database is completely empty and ready for fresh ingestion!")

    except mysql.connector.Error as err:
        print(f"\nDatabase Error encountered: {err}")
    finally:
        cur.close()
        conn.close()

# RUN THE WIPE PROCESS
wipe_entire_database()

⚠️ WARNING: This will permanently delete ALL DATA from database 'drugbank'.


Type WIPE ALL to confirm database clearing:  WIPE ALL


ProgrammingError: 1049 (42000): Unknown database 'drugbank'

<small>

### Stage 2 — Direct children of `drug` (FK → drug)

This stage loads the **tables that depend only on `drug_pk`** (foreign key to the `drug` table).

**Prerequisite:** Stage 1 must be completed (`drug` table populated), so every drug has a valid `drug_pk`.

---

#### Tables loaded in Stage 2

- `drugbank_id_map`
- `drug_group`
- `drug_classification`
- `drug_classification_alternative_parent`
- `drug_classification_substituent`
- `drug_synonym`
- `drug_salt`
- `product`
- `packager`
- `manufacturer`
- `drug_category`
- `drug_affected_organism`
- `drug_dosage`
- `drug_ahfs_code`
- `drug_pdb_entry`
- `drug_atc_code`
- `drug_atc_level` *(must run after `drug_atc_code`)*
- `drug_price`
- `drug_patent`
- `drug_food_interaction`
- `drug_interaction`
- `drug_external_identifier`
- `drug_external_link`
- `drug_property`
- `reaction`

---

#### Execution strategy (one XML scan per table)
This Stage 2 workflow runs **one full XML pass per table**, meaning:

- The XML file is re-read multiple times (once for each table).
- Each table gets its own:
  - progress bar
  - periodic status logs
  - commit batching

This is **slower** than a single-pass multi-table insert, but it is useful for debugging and isolating issues.

---

#### Dependency / sequencing
Inside Stage 2, the most important ordering rule is:

- **`drug_atc_code` must be loaded before `drug_atc_level`**

Everything else depends only on `drug_pk` and can be loaded independently.

---

#### Re-run safety and “skip if already loaded”
Some tables are naturally **idempotent** because the loader uses `INSERT IGNORE` or `ON DUPLICATE KEY UPDATE`
(e.g., many mapping tables).

Other tables can **duplicate on reruns** because they have no natural unique key or use auto-increment IDs
(e.g., `drug_salt`, `product`, `drug_price`, `drug_patent`, `drug_property`, `reaction`).

To avoid accidental duplicates:
- the Stage 2 runner can **skip a table** if it already has rows, OR
- explicitly **wipe (DELETE)** and reload selected tables.

---

#### Output / reporting
For each table pass, the runner reports:
- progress bar over total drugs in DB
- attempted vs affected rows
- per-table time spent

At the end, it prints a consolidated **summary table**:
- Table name
- Status (LOADED / SKIPPED)
- Attempted statements
- Affected rows (rowcount)
- Rows currently in DB
- Time taken

</small>


In [27]:
import mysql.connector

MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "niraj"
MYSQL_PASSWORD = "Niraj@123"
MYSQL_DB = "drugbank"

def fix_all_auto_increments():
    conn = mysql.connector.connect(
        host=MYSQL_HOST, port=MYSQL_PORT, user=MYSQL_USER, password=MYSQL_PASSWORD, database=MYSQL_DB, autocommit=True
    )
    cur = conn.cursor()
    
    # List of all primary key columns that MUST be AUTO_INCREMENT across all DrugBank tables
    pk_targets = [
        ("drug", "drug_pk"),
        ("drug_salt", "salt_pk"),
        ("drug_synonym", "synonym_pk"),
        ("drug_category", "category_pk"),
        ("drug_dosage", "dosage_pk"),
        ("drug_external_identifier", "dei_pk"),
        ("drug_external_link", "del_pk"),
        ("drug_food_interaction", "ffi_pk"),
        ("drug_patent", "patent_pk"),
        ("drug_price", "price_pk"),
        ("drug_property", "property_pk"),
        ("drug_affected_organism", "ao_pk"),
        ("interactant", "interactant_pk"),
        ("manufacturer", "manufacturer_pk"),
        ("packager", "packager_pk"),
        ("product", "product_pk"),
        ("reaction", "reaction_pk"),
        ("reference_item", "reference_pk"),
        ("polypeptide_external_identifier", "pei_pk"),
        ("polypeptide_synonym", "ps_pk"),
    ]

    print("🔧 Fixing AUTO_INCREMENT attributes across all tables...")
    cur.execute("SET FOREIGN_KEY_CHECKS=0;")
    
    fixed_count = 0
    for table, pk_col in pk_targets:
        try:
            # Check if table exists
            cur.execute(f"SHOW TABLES LIKE '{table}'")
            if not cur.fetchone():
                continue
                
            # Modify column to AUTO_INCREMENT
            cur.execute(f"ALTER TABLE `{table}` MODIFY `{pk_col}` BIGINT NOT NULL AUTO_INCREMENT;")
            print(f"  ✓ Fixed: {table}.{pk_col} -> AUTO_INCREMENT enabled")
            fixed_count += 1
        except mysql.connector.Error as err:
            print(f"  ⚠️ Notice on {table}.{pk_col}: {err}")

    cur.execute("SET FOREIGN_KEY_CHECKS=1;")
    cur.close()
    conn.close()
    print(f"\n🎉 Successfully updated {fixed_count} primary keys! You can now run Stage 2 without Error 1364.")

# Execute schema fix
fix_all_auto_increments()

🔧 Fixing AUTO_INCREMENT attributes across all tables...
  ✓ Fixed: drug.drug_pk -> AUTO_INCREMENT enabled
  ✓ Fixed: drug_salt.salt_pk -> AUTO_INCREMENT enabled
  ✓ Fixed: drug_synonym.synonym_pk -> AUTO_INCREMENT enabled
  ✓ Fixed: drug_category.category_pk -> AUTO_INCREMENT enabled
  ✓ Fixed: drug_dosage.dosage_pk -> AUTO_INCREMENT enabled
  ✓ Fixed: drug_external_identifier.dei_pk -> AUTO_INCREMENT enabled
  ✓ Fixed: drug_external_link.del_pk -> AUTO_INCREMENT enabled
  ✓ Fixed: drug_food_interaction.ffi_pk -> AUTO_INCREMENT enabled
  ✓ Fixed: drug_patent.patent_pk -> AUTO_INCREMENT enabled
  ✓ Fixed: drug_price.price_pk -> AUTO_INCREMENT enabled
  ✓ Fixed: drug_property.property_pk -> AUTO_INCREMENT enabled
  ✓ Fixed: drug_affected_organism.ao_pk -> AUTO_INCREMENT enabled
  ✓ Fixed: interactant.interactant_pk -> AUTO_INCREMENT enabled
  ✓ Fixed: manufacturer.manufacturer_pk -> AUTO_INCREMENT enabled
  ✓ Fixed: packager.packager_pk -> AUTO_INCREMENT enabled
  ✓ Fixed: product.produc

In [28]:
# =====================================================================================
# Stage 2 loader (ALL Stage-2 tables) - Fully Hardened against Data Truncation Errors
# =====================================================================================

import time
from dataclasses import dataclass
from typing import Optional, Dict, Any, Callable, Tuple, List

from lxml import etree as ET
from tqdm.notebook import tqdm
import mysql.connector

# ---------------------------
# CONFIGURATION
# ---------------------------
XML_PATH = r"db/drugbank.xml"

MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "niraj"
MYSQL_PASSWORD = "Niraj@123"
MYSQL_DB = "drugbank"

COMMIT_EVERY = 1000
SHOW_EVERY = 10000

SKIP_IF_NONEMPTY = True
FORCE_RELOAD_TABLES = set()
WIPE_BEFORE_LOAD_TABLES = set()
# ---------------------------

def get_conn():
    return mysql.connector.connect(
        host=MYSQL_HOST, port=MYSQL_PORT, user=MYSQL_USER,
        password=MYSQL_PASSWORD, database=MYSQL_DB, autocommit=False
    )

def count_table(conn, table: str) -> int:
    cur = conn.cursor()
    cur.execute(f"SELECT COUNT(*) FROM `{table}`")
    n = int(cur.fetchone()[0])
    cur.close()
    return n

def wipe_table_delete(table: str) -> None:
    conn = get_conn()
    cur = conn.cursor()
    cur.execute("SET FOREIGN_KEY_CHECKS=0;")
    cur.execute(f"DELETE FROM `{table}`")
    cur.execute("SET FOREIGN_KEY_CHECKS=1;")
    conn.commit()
    cur.close()
    conn.close()

# ---------------------------
# XML HELPERS
# ---------------------------
def strip_ns(tag: str) -> str:
    return tag.split("}", 1)[1] if "}" in tag else tag

def detect_namespace(xml_path: str) -> Optional[str]:
    ctx = ET.iterparse(xml_path, events=("start",))
    _, root = next(ctx)
    if root.tag.startswith("{"):
        return root.tag.split("}")[0].strip("{")
    return None

def nstag_factory(ns: Optional[str]):
    def nstag(local: str) -> str:
        return f"{{{ns}}}{local}" if ns else local
    return nstag

def text(elem) -> Optional[str]:
    if elem is None or elem.text is None:
        return None
    t = elem.text.strip()
    return t if t else None

def to_float(s: Optional[str]) -> Optional[float]:
    if s is None:
        return None
    try:
        return float(s)
    except:
        return None

def cleanup_lxml(elem):
    elem.clear()
    while elem.getprevious() is not None:
        del elem.getparent()[0]

def count_total_drugs_with_progress(xml_path: str) -> int:
    total = 0
    p = tqdm(total=None, unit="drug", dynamic_ncols=True, desc="Pre-pass: counting <drug>")
    try:
        ctx = ET.iterparse(xml_path, events=("end",), huge_tree=True)
        for _, e in ctx:
            if strip_ns(e.tag) == "drug":
                total += 1
                p.update(1)
                cleanup_lxml(e)
    finally:
        try: p.close()
        except: pass
    return total

def pick_primary_id(drug_elem, nstag) -> Optional[str]:
    ids = drug_elem.findall(nstag("drugbank-id"))
    first = None
    for ide in ids:
        did = text(ide)
        if not did:
            continue
        if first is None:
            first = did
        prim = ide.get("primary")
        if prim is not None:
            v = prim.strip().lower()
            if v not in ("false", "0", "no", "n", ""):
                return did
    return first

def load_drug_pk_map(conn) -> Dict[str, int]:
    cur = conn.cursor()
    cur.execute("SELECT primary_drugbank_id, drug_pk FROM drug")
    mp = {row[0]: int(row[1]) for row in cur.fetchall()}
    cur.close()
    return mp

@dataclass
class PassResult:
    table: str
    status: str
    attempted: int
    affected: int
    db_rows: int
    missing_drug: int
    seconds: float

def run_table_pass(
    xml_path: str,
    table: str,
    handler: Callable[[Any, int, Any, Callable[[str], str]], Tuple[int, int]],
    ns: Optional[str],
    drug_pk_map: Dict[str, int],
    total_drugs_xml: int,
) -> PassResult:
    nstag = nstag_factory(ns)

    conn_check = get_conn()
    current_rows = count_table(conn_check, table)
    conn_check.close()

    if table in FORCE_RELOAD_TABLES or table in WIPE_BEFORE_LOAD_TABLES:
        wipe_table_delete(table)
        current_rows = 0

    if SKIP_IF_NONEMPTY and current_rows > 0 and table not in FORCE_RELOAD_TABLES:
        return PassResult(table=table, status="SKIPPED (already has rows)", attempted=0, affected=0,
                          db_rows=current_rows, missing_drug=0, seconds=0.0)

    conn = get_conn()
    cur = conn.cursor()

    attempted_total = 0
    affected_total = 0
    missing_drug = 0
    processed = 0
    t0 = time.time()

    pbar = tqdm(total=total_drugs_xml, unit="drug", dynamic_ncols=True, desc=table)

    try:
        context = ET.iterparse(xml_path, events=("end",), huge_tree=True)
        for _, elem in context:
            if strip_ns(elem.tag) != "drug":
                continue

            primary_id = pick_primary_id(elem, nstag)
            if not primary_id:
                processed += 1
                pbar.update(1)
                cleanup_lxml(elem)
                continue

            drug_pk = drug_pk_map.get(primary_id)
            if drug_pk is None:
                missing_drug += 1
                processed += 1
                pbar.update(1)
                cleanup_lxml(elem)
                continue

            try:
                a, aff = handler(cur, drug_pk, elem, nstag)
                attempted_total += a
                affected_total += aff

                processed += 1
                pbar.update(1)

                if processed % COMMIT_EVERY == 0:
                    conn.commit()

            except Exception as e:
                conn.rollback()
                print(f"[{table}] ERROR drug_pk={drug_pk} ({primary_id}): {e}")

            finally:
                cleanup_lxml(elem)

        conn.commit()

    finally:
        try: pbar.close()
        except: pass
        cur.close()
        conn.close()

    seconds = time.time() - t0
    conn2 = get_conn()
    db_rows = count_table(conn2, table)
    conn2.close()

    return PassResult(table=table, status="LOADED", attempted=attempted_total, affected=affected_total,
                      db_rows=db_rows, missing_drug=missing_drug, seconds=seconds)

# =====================================================================================
# STAGE-2 HARDENED HANDLERS
# =====================================================================================

SQL_ID_MAP = """
INSERT INTO drugbank_id_map (drugbank_id, drug_pk, is_primary)
VALUES (%s,%s,%s)
ON DUPLICATE KEY UPDATE drug_pk=VALUES(drug_pk), is_primary=VALUES(is_primary)
"""
def h_drugbank_id_map(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    for ide in drug_elem.findall(nstag("drugbank-id")):
        did = text(ide)
        if not did:
            continue
        prim = ide.get("primary")
        is_primary = 1 if (prim and prim.strip().lower() not in ("false","0","no","n","")) else 0
        cur.execute(SQL_ID_MAP, (did[:32], drug_pk, is_primary))
        attempted += 1
        affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_GROUP = "INSERT IGNORE INTO drug_group (drug_pk, group_name) VALUES (%s,%s)"
def h_drug_group(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    groups = drug_elem.find(nstag("groups"))
    if groups is None:
        return 0, 0
    for g in groups.findall(nstag("group")):
        gv = text(g)
        if gv:
            cur.execute(SQL_GROUP, (drug_pk, gv[:128]))
            attempted += 1
            affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_CLASS = """
INSERT INTO drug_classification
  (drug_pk, description, direct_parent, kingdom, superclass, class_name, subclass_name)
VALUES (%s,%s,%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
  description=VALUES(description), direct_parent=VALUES(direct_parent),
  kingdom=VALUES(kingdom), superclass=VALUES(superclass),
  class_name=VALUES(class_name), subclass_name=VALUES(subclass_name)
"""
def h_drug_classification(cur, drug_pk, drug_elem, nstag):
    c = drug_elem.find(nstag("classification"))
    if c is None:
        return 0, 0
    def f(tag): return text(c.find(nstag(tag)))
    cur.execute(SQL_CLASS, (drug_pk, f("description"), 
                            f("direct-parent")[:512] if f("direct-parent") else None, 
                            f("kingdom")[:512] if f("kingdom") else None,
                            f("superclass")[:512] if f("superclass") else None, 
                            f("class")[:512] if f("class") else None, 
                            f("subclass")[:512] if f("subclass") else None))
    return 1, max(cur.rowcount, 0)

SQL_ALT_PARENT = "INSERT IGNORE INTO drug_classification_alternative_parent (drug_pk, value) VALUES (%s,%s)"
def h_alt_parent(cur, drug_pk, drug_elem, nstag):
    c = drug_elem.find(nstag("classification"))
    if c is None: return 0, 0
    aps = c.find(nstag("alternative-parents"))
    if aps is None: return 0, 0
    attempted = affected = 0
    for ap in aps.findall(nstag("alternative-parent")):
        v = text(ap)
        if v:
            cur.execute(SQL_ALT_PARENT, (drug_pk, v[:512]))
            attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_SUB = "INSERT IGNORE INTO drug_classification_substituent (drug_pk, value) VALUES (%s,%s)"
def h_substituent(cur, drug_pk, drug_elem, nstag):
    c = drug_elem.find(nstag("classification"))
    if c is None: return 0, 0
    subs = c.find(nstag("substituents"))
    if subs is None: return 0, 0
    attempted = affected = 0
    for s in subs.findall(nstag("substituent")):
        v = text(s)
        if v:
            cur.execute(SQL_SUB, (drug_pk, v[:512]))
            attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_SYNONYM = "INSERT IGNORE INTO drug_synonym (drug_pk, synonym, language, coder) VALUES (%s,%s,%s,%s)"
def h_drug_synonym(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    syns = drug_elem.find(nstag("synonyms"))
    if syns is None: return 0, 0
    for s in syns.findall(nstag("synonym")):
        sv = text(s)
        if sv:
            # Truncate synonym to 512 to prevent Data Too Long error
            cur.execute(SQL_SYNONYM, (drug_pk, sv[:512], (s.get("language") or "")[:32], (s.get("coder") or "")[:128]))
            attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_SALT = "INSERT INTO drug_salt (drug_pk, name, unii, cas_number, inchikey, average_mass, monoisotopic_mass) VALUES (%s,%s,%s,%s,%s,%s,%s)"
def h_drug_salt(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    salts = drug_elem.find(nstag("salts"))
    if salts is None: return 0, 0
    for salt in salts.findall(nstag("salt")):
        name = text(salt.find(nstag("name")))
        unii = text(salt.find(nstag("unii")))
        cas = text(salt.find(nstag("cas-number")))
        inchikey = text(salt.find(nstag("inchikey")))
        avg = to_float(text(salt.find(nstag("average-mass"))))
        mono = to_float(text(salt.find(nstag("monoisotopic-mass"))))
        cur.execute(SQL_SALT, (drug_pk, name[:512] if name else None, unii[:64] if unii else None, 
                               cas[:64] if cas else None, inchikey[:128] if inchikey else None, avg, mono))
        attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_PRODUCT = """
INSERT INTO product (
  drug_pk, name, labeller, ndc_id, ndc_product_code, dpd_id, ema_product_code,
  started_marketing_on, ended_marketing_on, dosage_form, strength, route,
  fda_application_number, generic, over_the_counter, approved, country, source
)
VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
"""
def h_product(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    products = drug_elem.find(nstag("products"))
    if products is None: return 0, 0
    for p in products.findall(nstag("product")):
        def f(tag): return text(p.find(nstag(tag)))
        generic = 1 if (f("generic") or "").strip().lower() in ("true","1","yes") else 0
        otc = 1 if (f("over-the-counter") or "").strip().lower() in ("true","1","yes") else 0
        approved = 1 if (f("approved") or "").strip().lower() in ("true","1","yes") else 0
        cur.execute(SQL_PRODUCT, (
            drug_pk, f("name")[:512] if f("name") else None, f("labeller")[:512] if f("labeller") else None,
            f("ndc-id")[:64] if f("ndc-id") else None, f("ndc-product-code")[:64] if f("ndc-product-code") else None,
            f("dpd-id")[:64] if f("dpd-id") else None, f("ema-product-code")[:64] if f("ema-product-code") else None,
            f("started-marketing-on"), f("ended-marketing-on"), f("dosage-form")[:256] if f("dosage-form") else None,
            f("strength")[:256] if f("strength") else None, f("route")[:128] if f("route") else None,
            f("fda-application-number")[:64] if f("fda-application-number") else None, generic, otc, approved,
            f("country")[:128] if f("country") else None, f("source")[:128] if f("source") else None
        ))
        attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_PACKAGER = "INSERT INTO packager (drug_pk, name, url) VALUES (%s,%s,%s)"
def h_packager(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    packagers = drug_elem.find(nstag("packagers"))
    if packagers is None: return 0, 0
    for p in packagers.findall(nstag("packager")):
        name = text(p.find(nstag("name")))
        url = text(p.find(nstag("url")))
        if name:
            cur.execute(SQL_PACKAGER, (drug_pk, name[:512], url))
            attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_MANUFACTURER = "INSERT INTO manufacturer (drug_pk, name, generic, url) VALUES (%s,%s,%s,%s)"
def h_manufacturer(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    mfgs = drug_elem.find(nstag("manufacturers"))
    if mfgs is None: return 0, 0
    for m in mfgs.findall(nstag("manufacturer")):
        name = text(m)
        if not name: continue
        generic_attr = (m.get("generic") or "").strip().lower()
        generic = 1 if generic_attr in ("true","1","yes") else 0
        url = m.get("url")
        cur.execute(SQL_MANUFACTURER, (drug_pk, name[:512], generic, url))
        attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_CATEGORY = "INSERT IGNORE INTO drug_category (drug_pk, category, mesh_id) VALUES (%s,%s,%s)"
def h_drug_category(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    cats = drug_elem.find(nstag("categories"))
    if cats is None: return 0, 0
    for c in cats.findall(nstag("category")):
        cat = text(c.find(nstag("category")))
        mesh = text(c.find(nstag("mesh-id")))
        if cat:
            cur.execute(SQL_CATEGORY, (drug_pk, cat[:512], mesh[:64] if mesh else None))
            attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_AO = "INSERT IGNORE INTO drug_affected_organism (drug_pk, organism) VALUES (%s,%s)"
def h_affected_organism(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    aos = drug_elem.find(nstag("affected-organisms"))
    if aos is None: return 0, 0
    for ao in aos.findall(nstag("affected-organism")):
        v = text(ao)
        if v:
            cur.execute(SQL_AO, (drug_pk, v[:512]))
            attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_DOSAGE = "INSERT IGNORE INTO drug_dosage (drug_pk, form, route, strength) VALUES (%s,%s,%s,%s)"
def h_drug_dosage(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    dosages = drug_elem.find(nstag("dosages"))
    if dosages is None: return 0, 0
    for d in dosages.findall(nstag("dosage")):
        form = text(d.find(nstag("form")))
        route = text(d.find(nstag("route")))
        strength = text(d.find(nstag("strength")))
        cur.execute(SQL_DOSAGE, (drug_pk, form[:256] if form else None, route[:128] if route else None, strength[:256] if strength else None))
        attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_AHFS = "INSERT IGNORE INTO drug_ahfs_code (drug_pk, ahfs_code) VALUES (%s,%s)"
def h_drug_ahfs(cur, drug_pk, drug_elem, nstag):
    ahfs = drug_elem.find(nstag("ahfs-codes"))
    if ahfs is None: return 0, 0
    attempted = affected = 0
    for child in list(ahfs):
        v = text(child)
        if v:
            cur.execute(SQL_AHFS, (drug_pk, v[:64]))
            attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_PDB = "INSERT IGNORE INTO drug_pdb_entry (drug_pk, pdb_entry) VALUES (%s,%s)"
def h_drug_pdb(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    pdbs = drug_elem.find(nstag("pdb-entries"))
    if pdbs is None: return 0, 0
    for p in pdbs.findall(nstag("pdb-entry")):
        v = text(p)
        if v:
            cur.execute(SQL_PDB, (drug_pk, v[:32]))
            attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_ATC_CODE = "INSERT IGNORE INTO drug_atc_code (drug_pk, atc_code) VALUES (%s,%s)"
def h_drug_atc_code(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    atcs = drug_elem.find(nstag("atc-codes"))
    if atcs is None: return 0, 0
    for atc in atcs.findall(nstag("atc-code")):
        code = atc.get("code")
        if code:
            cur.execute(SQL_ATC_CODE, (drug_pk, code[:16]))
            attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_ATC_LEVEL = """
INSERT INTO drug_atc_level (drug_pk, atc_code, level_no, level_text, level_code_attr)
VALUES (%s,%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE level_text=VALUES(level_text), level_code_attr=VALUES(level_code_attr)
"""
def h_drug_atc_level(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    atcs = drug_elem.find(nstag("atc-codes"))
    if atcs is None: return 0, 0
    for atc in atcs.findall(nstag("atc-code")):
        code = atc.get("code")
        if not code: continue
        for i, lvl in enumerate(atc.findall(nstag("level")), start=1):
            cur.execute(SQL_ATC_LEVEL, (drug_pk, code[:16], i, text(lvl)[:512] if text(lvl) else None, lvl.get("code")))
            attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_PRICE = "INSERT INTO drug_price (drug_pk, description, cost_value, cost_currency, unit) VALUES (%s,%s,%s,%s,%s)"
def h_drug_price(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    prices = drug_elem.find(nstag("prices"))
    if prices is None: return 0, 0
    for price in prices.findall(nstag("price")):
        desc = text(price.find(nstag("description")))
        cost_elem = price.find(nstag("cost"))
        cost_value = text(cost_elem)
        cost_currency = cost_elem.get("currency") if cost_elem is not None else None
        unit = text(price.find(nstag("unit")))
        cur.execute(SQL_PRICE, (drug_pk, desc[:512] if desc else None, cost_value[:64] if cost_value else None, 
                               cost_currency[:16] if cost_currency else None, unit[:64] if unit else None))
        attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_PATENT = "INSERT INTO drug_patent (drug_pk, number, country, approved, expires, pediatric_extension) VALUES (%s,%s,%s,%s,%s,%s)"
def h_drug_patent(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    pats = drug_elem.find(nstag("patents"))
    if pats is None: return 0, 0
    for pat in pats.findall(nstag("patent")):
        ped = 1 if (text(pat.find(nstag("pediatric-extension"))) or "").strip().lower() in ("true","1","yes") else 0
        cur.execute(SQL_PATENT, (drug_pk, text(pat.find(nstag("number"))), text(pat.find(nstag("country"))),
                                text(pat.find(nstag("approved"))), text(pat.find(nstag("expires"))), ped))
        attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_FOOD = "INSERT IGNORE INTO drug_food_interaction (drug_pk, interaction_text) VALUES (%s,%s)"
def h_drug_food(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    food = drug_elem.find(nstag("food-interactions"))
    if food is None: return 0, 0
    for fi in food.findall(nstag("food-interaction")):
        v = text(fi)
        if v:
            cur.execute(SQL_FOOD, (drug_pk, v))
            attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_DRUG_INTERACTION = """
INSERT INTO drug_interaction (drug_pk, interacting_drugbank_id, interacting_name, description)
VALUES (%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE interacting_name=VALUES(interacting_name), description=VALUES(description)
"""
def h_drug_interaction(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    dis = drug_elem.find(nstag("drug-interactions"))
    if dis is None: return 0, 0
    for di in dis.findall(nstag("drug-interaction")):
        other_id = text(di.find(nstag("drugbank-id")))
        if not other_id: continue
        cur.execute(SQL_DRUG_INTERACTION, (drug_pk, other_id[:32], text(di.find(nstag("name"))), text(di.find(nstag("description")))))
        attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_EXT_ID = "INSERT IGNORE INTO drug_external_identifier (drug_pk, resource, identifier) VALUES (%s,%s,%s)"
def h_ext_id(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    ex_ids = drug_elem.find(nstag("external-identifiers"))
    if ex_ids is None: return 0, 0
    for exi in ex_ids.findall(nstag("external-identifier")):
        resource = text(exi.find(nstag("resource")))
        identifier = text(exi.find(nstag("identifier")))
        if resource and identifier:
            cur.execute(SQL_EXT_ID, (drug_pk, resource[:256], identifier[:256]))
            attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_EXT_LINK = "INSERT IGNORE INTO drug_external_link (drug_pk, resource, url) VALUES (%s,%s,%s)"
def h_ext_link(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    ex_links = drug_elem.find(nstag("external-links"))
    if ex_links is None: return 0, 0
    for exl in ex_links.findall(nstag("external-link")):
        resource = text(exl.find(nstag("resource")))
        url = text(exl.find(nstag("url")))
        if resource and url:
            cur.execute(SQL_EXT_LINK, (drug_pk, resource[:256], url))
            attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

SQL_PROPERTY = "INSERT INTO drug_property (drug_pk, property_type, kind, value, source) VALUES (%s,%s,%s,%s,%s)"
def h_drug_property(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    def insert_props(container_tag: str, prop_type: str):
        nonlocal attempted, affected
        cont = drug_elem.find(nstag(container_tag))
        if cont is None: return
        for prop in cont.findall(nstag("property")):
            cur.execute(SQL_PROPERTY, (drug_pk, prop_type,
                                       text(prop.find(nstag("kind")))[:256] if text(prop.find(nstag("kind"))) else None,
                                       text(prop.find(nstag("value")))[:512] if text(prop.find(nstag("value"))) else None,
                                       text(prop.find(nstag("source")))[:256] if text(prop.find(nstag("source"))) else None))
            attempted += 1; affected += max(cur.rowcount, 0)
    insert_props("calculated-properties", "calculated")
    insert_props("experimental-properties", "experimental")
    return attempted, affected

SQL_REACTION = "INSERT INTO reaction (drug_pk, sequence_text) VALUES (%s,%s)"
def h_reaction(cur, drug_pk, drug_elem, nstag):
    attempted = affected = 0
    reactions = drug_elem.find(nstag("reactions"))
    if reactions is None: return 0, 0
    for rxn in reactions.findall(nstag("reaction")):
        cur.execute(SQL_REACTION, (drug_pk, text(rxn.find(nstag("sequence")))))
        attempted += 1; affected += max(cur.rowcount, 0)
    return attempted, affected

# ---------------------------
# EXECUTION
# ---------------------------
print("Stage 2 pre-pass: counting total <drug> in XML...")
total_drugs_xml = count_total_drugs_with_progress(XML_PATH)
print(f"Total drugs in XML: {total_drugs_xml:,}")

ns = detect_namespace(XML_PATH)
conn0 = get_conn()
drug_pk_map = load_drug_pk_map(conn0)
conn0.close()

print(f"Drugs in DB (Stage 1): {len(drug_pk_map):,}")

tasks = [
    ("drugbank_id_map", h_drugbank_id_map),
    ("drug_group", h_drug_group),
    ("drug_classification", h_drug_classification),
    ("drug_classification_alternative_parent", h_alt_parent),
    ("drug_classification_substituent", h_substituent),
    ("drug_synonym", h_drug_synonym),
    ("drug_salt", h_drug_salt),
    ("product", h_product),
    ("packager", h_packager),
    ("manufacturer", h_manufacturer),
    ("drug_category", h_drug_category),
    ("drug_affected_organism", h_affected_organism),
    ("drug_dosage", h_drug_dosage),
    ("drug_ahfs_code", h_drug_ahfs),
    ("drug_pdb_entry", h_drug_pdb),
    ("drug_atc_code", h_drug_atc_code),
    ("drug_atc_level", h_drug_atc_level),
    ("drug_price", h_drug_price),
    ("drug_patent", h_drug_patent),
    ("drug_food_interaction", h_drug_food),
    ("drug_interaction", h_drug_interaction),
    ("drug_external_identifier", h_ext_id),
    ("drug_external_link", h_ext_link),
    ("drug_property", h_drug_property),
    ("reaction", h_reaction),
]

results: List[PassResult] = []
for table, handler in tasks:
    results.append(run_table_pass(XML_PATH, table, handler, ns, drug_pk_map, total_drugs_xml))

print("\n=== Stage 2 Summary (Hardened) ===")
for r in results:
    print(f"{r.table:38} | {r.status:25} | Rows in DB: {r.db_rows:,}")

Stage 2 pre-pass: counting total <drug> in XML...


Pre-pass: counting <drug>: 0drug [00:00, ?drug/s]

Total drugs in XML: 73,687
Drugs in DB (Stage 1): 17,430


drugbank_id_map:   0%|          | 0/73687 [00:00<?, ?drug/s]

drug_group:   0%|          | 0/73687 [00:00<?, ?drug/s]

drug_classification:   0%|          | 0/73687 [00:00<?, ?drug/s]

... (19 outputs left out)

drug_external_link:   0%|          | 0/73687 [00:00<?, ?drug/s]

drug_property:   0%|          | 0/73687 [00:00<?, ?drug/s]

reaction:   0%|          | 0/73687 [00:00<?, ?drug/s]


=== Stage 2 Summary (Hardened) ===
drugbank_id_map                        | LOADED                    | Rows in DB: 22,428
drug_group                             | LOADED                    | Rows in DB: 20,354
drug_classification                    | LOADED                    | Rows in DB: 11,154
drug_classification_alternative_parent | LOADED                    | Rows in DB: 0
drug_classification_substituent        | LOADED                    | Rows in DB: 0
drug_synonym                           | LOADED                    | Rows in DB: 39,982
drug_salt                              | LOADED                    | Rows in DB: 2,922
product                                | LOADED                    | Rows in DB: 462,594
packager                               | LOADED                    | Rows in DB: 20,499
manufacturer                           | LOADED                    | Rows in DB: 5,971
drug_category                          | LOADED                    | Rows in DB: 101,512
drug_a

<small>

### Stage 3 — Children of `drug_salt` (drug_salt_drugbank_id)

This stage inserts into:

- **`drug_salt_drugbank_id`** *(Stage 3)* — links **salt** → **DrugBank salt IDs**  
  Foreign key: `drug_salt_drugbank_id.salt_pk` → `drug_salt.salt_pk`

---

#### Prerequisites
Before running Stage 3, you must have:

- **Stage 1:** `drug` loaded (so `drug_pk` exists)
- **Stage 2:** `drug_salt` loaded (so `salt_pk` exists)

---

#### What the Stage 3 cell does
- Streams the XML once
- For each `<drug>`:
  - Resolves `drug_pk` (from `primary_drugbank_id`)
  - Reads `<salts>/<salt>` entries
  - Matches each XML `<salt>` to the corresponding DB row in `drug_salt`
  - Inserts each `<salt>/<drugbank-id>` into `drug_salt_drugbank_id`

---

#### Matching logic (XML salt → DB salt)
To find the correct `salt_pk`, it matches the salt by these fields:

- `name`
- `unii`
- `cas-number`
- `inchikey`
- `average-mass`
- `monoisotopic-mass`

Because floating values can differ slightly, the code normalizes/rounds numeric values to reduce mismatch risk.

If a salt cannot be matched, it is counted as **unmatched_salts**.

---

#### Insert behavior
- Uses **`INSERT IGNORE`** into `drug_salt_drugbank_id`  
  → safe to re-run without duplicating rows.

Optional settings:
- **Skip if non-empty:** if `drug_salt_drugbank_id` already has rows, the stage can be skipped.
- **Wipe before load:** can `DELETE FROM drug_salt_drugbank_id` (safe; it’s a child table).

---

#### Output / reporting
The stage prints:
- Progress bar over total drugs
- Periodic status (attempted inserts, inserted rows, matched/unmatched salts)
- Final summary table:
  - attempted vs inserted rows
  - rows in DB
  - matched/unmatched salt counts
  - errors and runtime

</small>


In [29]:
import time
from dataclasses import dataclass
from typing import Optional, Dict, Any, Tuple, List

from lxml import etree as ET
from tqdm.notebook import tqdm
import mysql.connector

# ---------------------------
# CONFIGURATION
# ---------------------------
XML_PATH = r"db/drugbank.xml"

MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "niraj"
MYSQL_PASSWORD = "Niraj@123"
MYSQL_DB = "drugbank"

COMMIT_EVERY = 2000
SHOW_EVERY = 10000

SKIP_IF_NONEMPTY = True
WIPE_BEFORE_LOAD = False
# ---------------------------

def get_conn():
    return mysql.connector.connect(
        host=MYSQL_HOST,
        port=MYSQL_PORT,
        user=MYSQL_USER,
        password=MYSQL_PASSWORD,
        database=MYSQL_DB,
        autocommit=False,
    )

def count_table(conn, table: str) -> int:
    cur = conn.cursor()
    cur.execute(f"SELECT COUNT(*) FROM `{table}`")
    n = int(cur.fetchone()[0])
    cur.close()
    return n

def strip_ns(tag: str) -> str:
    return tag.split("}", 1)[1] if "}" in tag else tag

def detect_namespace(xml_path: str) -> Optional[str]:
    ctx = ET.iterparse(xml_path, events=("start",))
    _, root = next(ctx)
    if root.tag.startswith("{"):
        return root.tag.split("}")[0].strip("{")
    return None

def nstag_factory(ns: Optional[str]):
    def nstag(local: str) -> str:
        return f"{{{ns}}}{local}" if ns else local
    return nstag

def text(elem) -> Optional[str]:
    if elem is None or elem.text is None:
        return None
    t = elem.text.strip()
    return t if t else None

def bool_from_text(s: Optional[str]) -> bool:
    if s is None:
        return False
    v = s.strip().lower()
    return v in ("true", "1", "yes")

def to_float(s: Optional[str]) -> Optional[float]:
    if s is None:
        return None
    try:
        return float(s)
    except:
        return None

def cleanup_lxml(elem):
    elem.clear()
    while elem.getprevious() is not None:
        del elem.getparent()[0]

def get_primary_drugbank_id(drug_elem, nstag) -> Optional[str]:
    ids = drug_elem.findall(nstag("drugbank-id"))
    fallback = None
    primary = None
    for ide in ids:
        did = text(ide)
        if not did:
            continue
        if fallback is None:
            fallback = did
        if bool_from_text(ide.get("primary")):
            primary = did
            break
    return primary or fallback

def load_drug_pk_map(conn) -> Dict[str, int]:
    cur = conn.cursor()
    cur.execute("SELECT primary_drugbank_id, drug_pk FROM drug")
    mp = {row[0]: int(row[1]) for row in cur.fetchall()}
    cur.close()
    return mp

def norm_str(s: Optional[str], length: int = 512) -> Optional[str]:
    if isinstance(s, str) and s.strip():
        return s.strip()[:length]
    return None

def norm_float(v: Optional[float]) -> Optional[float]:
    if v is None:
        return None
    return round(float(v), 6)

def salt_key(name, unii, cas, inchikey, avg, mono) -> Tuple:
    return (
        norm_str(name),
        norm_str(unii, 64),
        norm_str(cas, 64),
        norm_str(inchikey, 128),
        norm_float(avg),
        norm_float(mono),
    )

SQL_LOAD_SALTS_FOR_DRUG = """
SELECT salt_pk, name, unii, cas_number, inchikey, average_mass, monoisotopic_mass
FROM drug_salt
WHERE drug_pk = %s
ORDER BY salt_pk
"""

SQL_INSERT_SALT_ID = """
INSERT IGNORE INTO drug_salt_drugbank_id (salt_pk, drugbank_id, is_primary)
VALUES (%s, %s, %s)
"""

@dataclass
class Stage3Result:
    table: str
    status: str
    attempted: int
    inserted: int
    rows_in_db: int
    processed_drugs: int
    matched_salts: int
    unmatched_salts: int
    missing_drug: int
    errors: int
    seconds: float

def import_stage3_drug_salt_drugbank_id(xml_path: str) -> Stage3Result:
    ns = detect_namespace(xml_path)
    nstag = nstag_factory(ns)

    # Pre-check
    conn_check = get_conn()
    existing = count_table(conn_check, "drug_salt_drugbank_id")
    conn_check.close()

    if SKIP_IF_NONEMPTY and existing > 0 and not WIPE_BEFORE_LOAD:
        return Stage3Result(
            table="drug_salt_drugbank_id",
            status="SKIPPED (already has rows)",
            attempted=0, inserted=0, rows_in_db=existing,
            processed_drugs=0, matched_salts=0, unmatched_salts=0,
            missing_drug=0, errors=0, seconds=0.0
        )

    if WIPE_BEFORE_LOAD:
        conn_wipe = get_conn()
        cur_wipe = conn_wipe.cursor()
        cur_wipe.execute("SET FOREIGN_KEY_CHECKS=0;")
        cur_wipe.execute("DELETE FROM drug_salt_drugbank_id")
        cur_wipe.execute("SET FOREIGN_KEY_CHECKS=1;")
        conn_wipe.commit()
        cur_wipe.close()
        conn_wipe.close()

    conn = get_conn()
    cur = conn.cursor()

    drug_pk_map = load_drug_pk_map(conn)
    total_drugs = len(drug_pk_map)
    if total_drugs == 0:
        cur.close()
        conn.close()
        raise RuntimeError("drug table is empty. Run Stage 1 first.")

    attempted = 0
    inserted = 0
    processed_drugs = 0
    matched_salts = 0
    unmatched_salts = 0
    missing_drug = 0
    errors = 0

    salts_cache: Dict[int, Dict[Tuple, List[int]]] = {}

    def get_salts_map_for_drug(drug_pk: int) -> Dict[Tuple, List[int]]:
        if drug_pk in salts_cache:
            return salts_cache[drug_pk]

        cur.execute(SQL_LOAD_SALTS_FOR_DRUG, (drug_pk,))
        rows = cur.fetchall()

        m: Dict[Tuple, List[int]] = {}
        for (salt_pk, name, unii, cas, inchikey, avg, mono) in rows:
            k = salt_key(name, unii, cas, inchikey, avg, mono)
            m.setdefault(k, []).append(int(salt_pk))

        salts_cache[drug_pk] = m
        return m

    t0 = time.time()
    pbar = tqdm(total=total_drugs, unit="drug", dynamic_ncols=True, desc="drug_salt_drugbank_id (Stage 3)")

    try:
        context = ET.iterparse(xml_path, events=("end",), huge_tree=True)
        for _, elem in context:
            if strip_ns(elem.tag) != "drug":
                continue

            primary_id = get_primary_drugbank_id(elem, nstag)
            if not primary_id:
                cleanup_lxml(elem)
                continue

            drug_pk = drug_pk_map.get(primary_id)
            if drug_pk is None:
                missing_drug += 1
                cleanup_lxml(elem)
                continue

            try:
                salts = elem.find(nstag("salts"))
                if salts is not None:
                    salts_map = get_salts_map_for_drug(drug_pk)

                    for salt in salts.findall(nstag("salt")):
                        s_name = text(salt.find(nstag("name")))
                        s_unii = text(salt.find(nstag("unii")))
                        s_cas = text(salt.find(nstag("cas-number")))
                        s_inchikey = text(salt.find(nstag("inchikey")))
                        s_avg = to_float(text(salt.find(nstag("average-mass"))))
                        s_mono = to_float(text(salt.find(nstag("monoisotopic-mass"))))

                        k = salt_key(s_name, s_unii, s_cas, s_inchikey, s_avg, s_mono)
                        lst = salts_map.get(k, [])
                        if not lst:
                            unmatched_salts += 1
                            continue

                        salt_pk = lst.pop(0)
                        matched_salts += 1

                        for sid in salt.findall(nstag("drugbank-id")):
                            did = text(sid)
                            if not did:
                                continue
                            is_primary = 1 if bool_from_text(sid.get("primary")) else 0
                            
                            # CRITICAL FIX: Explicitly cap did to 32 characters
                            cur.execute(SQL_INSERT_SALT_ID, (salt_pk, did[:32], is_primary))
                            attempted += 1
                            inserted += max(cur.rowcount, 0)

                processed_drugs += 1
                pbar.update(1)

                if processed_drugs % COMMIT_EVERY == 0:
                    conn.commit()

            except Exception as e:
                errors += 1
                conn.rollback()
                print(f"[Stage3] ERROR drug_pk={drug_pk} ({primary_id}): {e}")

            finally:
                cleanup_lxml(elem)

        conn.commit()

    finally:
        try: pbar.close()
        except Exception: pass
        cur.close()
        conn.close()

    seconds = time.time() - t0
    conn2 = get_conn()
    rows_in_db = count_table(conn2, "drug_salt_drugbank_id")
    conn2.close()

    return Stage3Result(
        table="drug_salt_drugbank_id",
        status="LOADED",
        attempted=attempted,
        inserted=inserted,
        rows_in_db=rows_in_db,
        processed_drugs=processed_drugs,
        matched_salts=matched_salts,
        unmatched_salts=unmatched_salts,
        missing_drug=missing_drug,
        errors=errors,
        seconds=seconds,
    )

def print_result(r: Stage3Result):
    headers = ["Table", "Status", "Attempted", "Inserted", "Rows in DB", "Processed drugs", "Matched salts", "Unmatched salts", "Missing drug", "Errors", "Time"]
    rows = [[
        r.table, r.status, f"{r.attempted:,}", f"{r.inserted:,}", f"{r.rows_in_db:,}",
        f"{r.processed_drugs:,}", f"{r.matched_salts:,}", f"{r.unmatched_salts:,}",
        f"{r.missing_drug:,}", f"{r.errors:,}", f"{r.seconds:.1f}s",
    ]]
    widths = [len(h) for h in headers]
    for row in rows:
        for i, v in enumerate(row):
            widths[i] = max(widths[i], len(str(v)))
    def sep(ch="-"): return "+".join(ch * (w + 2) for w in widths)
    def render(row): return "|".join(" " + str(row[i]).ljust(widths[i]) + " " for i in range(len(headers)))
    print(sep("-"))
    print(render(headers))
    print(sep("="))
    print(render(rows[0]))
    print(sep("-"))

# RUN STAGE 3
result = import_stage3_drug_salt_drugbank_id(XML_PATH)
print_result(result)

drug_salt_drugbank_id (Stage 3):   0%|          | 0/17430 [00:00<?, ?drug/s]

-----------------------+--------+-----------+----------+------------+-----------------+---------------+-----------------+--------------+--------+-------
 Table                 | Status | Attempted | Inserted | Rows in DB | Processed drugs | Matched salts | Unmatched salts | Missing drug | Errors | Time  
=======================+========+===========+==========+============+=================+===============+=================+==============+========+=======
 drug_salt_drugbank_id | LOADED | 2,922     | 2,922    | 2,922      | 73,687          | 2,922         | 0               | 0            | 0      | 64.1s 
-----------------------+--------+-----------+----------+------------+-----------------+---------------+-----------------+--------------+--------+-------


<small>

### Stage 4 — References link tables (drug_reference, interactant_reference)

This stage builds the **reference linking tables** using foreign keys:

- **`reference_item`** *(Stage 0 prerequisite)* — inserted **as needed**
- **`drug_reference`** *(Stage 4)* — links `drug` → `reference_item`
- **`interactant_reference`** *(Stage 4)* — links `interactant` → `reference_item`  
  *(Only runs if the `interactant` table is already loaded.)*

---

#### What this Stage 4 cell does

**1) Count total drugs first (pre-pass)**  
- Performs a **streaming pre-pass** over the XML and counts total `<drug>` elements.  
- Uses this number for a **true progress bar** in the main pass.

**2) Main processing pass (with progress)**  
- Streams the XML again and processes each `<drug>`:
  - Inserts rows into **`reference_item`** if needed
  - Inserts rows into **`drug_reference`**
  - Optionally inserts rows into **`interactant_reference`**

---

#### Matching & deduplication logic

**A) `reference_item` dedupe**  
`reference_item` has **no natural unique key**, so we deduplicate using:

- Python cache + DB lookup using a composite key:  
  `(ref_type, ref_id, pubmed_id, citation, isbn, title, url)`

**B) `drug_reference` insertion**  
- Uses `INSERT IGNORE` into `(drug_pk, reference_pk)`.

**C) `interactant_reference` insertion (only if interactants exist)**  
Requires `interactant` rows already loaded (Stage 7). We locate `interactant_pk` by matching:

- `drug_pk + kind + position + interactant_id + name + organism + known_action`

If not found, the link is skipped and counted as **missing_interactant**.

---

#### Notes
- This implementation is **correctness-first**, not speed-first.
- It runs the XML twice (one pass for counting, one pass for insertion).

</small>


In [31]:
# =============================================================================
# STAGE 4 (Jupyter single cell) - HARDENED & CRASH-PROOF
# Builds:
#   - reference_item       (Deduplicated via in-memory cache & DB lookup)
#   - drug_reference       (Stage 4)
#   - interactant_reference (Stage 4, ONLY if interactant table is loaded)
# =============================================================================

import time
from dataclasses import dataclass
from typing import Optional, Dict, Tuple, Any, List

from lxml import etree as ET
from tqdm.notebook import tqdm
import mysql.connector

# ---------------------------
# CONFIGURATION
# ---------------------------
XML_PATH = r"db/drugbank.xml"

MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "niraj"
MYSQL_PASSWORD = "Niraj@123"
MYSQL_DB = "drugbank"

COMMIT_EVERY = 1000
SHOW_EVERY = 10000

# Set to False if interactant table is not populated yet
DO_INTERACTANT_REFERENCE = True
# ---------------------------


def get_conn():
    return mysql.connector.connect(
        host=MYSQL_HOST,
        port=MYSQL_PORT,
        user=MYSQL_USER,
        password=MYSQL_PASSWORD,
        database=MYSQL_DB,
        autocommit=False,
    )

def strip_ns(tag: str) -> str:
    return tag.split("}", 1)[1] if "}" in tag else tag

def detect_namespace(xml_path: str) -> Optional[str]:
    ctx = ET.iterparse(xml_path, events=("start",))
    _, root = next(ctx)
    if root.tag.startswith("{"):
        return root.tag.split("}")[0].strip("{")
    return None

def nstag_factory(ns: Optional[str]):
    def nstag(local: str) -> str:
        return f"{{{ns}}}{local}" if ns else local
    return nstag

def text(elem) -> Optional[str]:
    if elem is None or elem.text is None:
        return None
    t = elem.text.strip()
    return t if t else None

def bool_from_text(s: Optional[str]) -> bool:
    if s is None:
        return False
    v = s.strip().lower()
    return v in ("true", "1", "yes")

def cleanup_lxml(elem):
    elem.clear()
    while elem.getprevious() is not None:
        del elem.getparent()[0]

def get_primary_drugbank_id(drug_elem, nstag) -> Optional[str]:
    ids = drug_elem.findall(nstag("drugbank-id"))
    fallback = None
    primary = None
    for ide in ids:
        did = text(ide)
        if not did:
            continue
        if fallback is None:
            fallback = did
        if bool_from_text(ide.get("primary")):
            primary = did
            break
    return primary or fallback

def load_drug_pk_map(conn) -> Dict[str, int]:
    cur = conn.cursor()
    cur.execute("SELECT primary_drugbank_id, drug_pk FROM drug")
    mp = {row[0]: int(row[1]) for row in cur.fetchall()}
    cur.close()
    return mp

def table_exists(conn, table: str) -> bool:
    cur = conn.cursor()
    cur.execute(
        "SELECT COUNT(*) FROM information_schema.tables "
        "WHERE table_schema=%s AND table_name=%s",
        (MYSQL_DB, table),
    )
    ok = int(cur.fetchone()[0]) == 1
    cur.close()
    return ok

def count_table(conn, table: str) -> int:
    cur = conn.cursor()
    cur.execute(f"SELECT COUNT(*) FROM `{table}`")
    n = int(cur.fetchone()[0])
    cur.close()
    return n

def count_total_drugs_in_xml(xml_path: str) -> int:
    total = 0
    ctx = ET.iterparse(xml_path, events=("end",), huge_tree=True)
    for _, elem in ctx:
        if strip_ns(elem.tag) == "drug":
            total += 1
            cleanup_lxml(elem)
    return total

# ---------------------------
# OPTIMIZED SQL STATEMENTS
# ---------------------------
SQL_REF_SELECT = """
SELECT reference_pk
FROM reference_item
WHERE ref_type <=> %s
  AND ref_id <=> %s
  AND pubmed_id <=> %s
LIMIT 1
"""

SQL_REF_INSERT = """
INSERT INTO reference_item
  (ref_type, ref_id, pubmed_id, citation, isbn, title, url)
VALUES (%s,%s,%s,%s,%s,%s,%s)
"""

SQL_DRUG_REF_INSERT = "INSERT IGNORE INTO drug_reference (drug_pk, reference_pk) VALUES (%s,%s)"

SQL_INTERACTANT_FIND = """
SELECT interactant_pk
FROM interactant
WHERE drug_pk=%s
  AND kind <=> %s
  AND interactant_id <=> %s
ORDER BY interactant_pk DESC
LIMIT 1
"""

SQL_INTERACTANT_REF_INSERT = "INSERT IGNORE INTO interactant_reference (interactant_pk, reference_pk) VALUES (%s,%s)"

# ---------------------------
# REFERENCE PARSING WITH TRUNCATION
# ---------------------------
def parse_reference_list(ref_list_elem, nstag) -> List[Tuple[str, Optional[str], Optional[str], Optional[str], Optional[str], Optional[str], Optional[str]]]:
    out = []

    # articles/article => ref-id, pubmed-id, citation
    arts = ref_list_elem.find(nstag("articles"))
    if arts is not None:
        for a in arts.findall(nstag("article")):
            ref_id = text(a.find(nstag("ref-id")))
            pubmed_id = text(a.find(nstag("pubmed-id")))
            citation = text(a.find(nstag("citation")))
            out.append((
                "article",
                ref_id[:128] if ref_id else None,
                pubmed_id[:32] if pubmed_id else None,
                citation,
                None, None, None
            ))

    # textbooks/textbook => ref-id, isbn, citation
    tbs = ref_list_elem.find(nstag("textbooks"))
    if tbs is not None:
        for t in tbs.findall(nstag("textbook")):
            ref_id = text(t.find(nstag("ref-id")))
            isbn = text(t.find(nstag("isbn")))
            citation = text(t.find(nstag("citation")))
            out.append((
                "textbook",
                ref_id[:128] if ref_id else None,
                None,
                citation,
                isbn[:64] if isbn else None,
                None, None
            ))

    # links/link => ref-id, title, url
    links = ref_list_elem.find(nstag("links"))
    if links is not None:
        for l in links.findall(nstag("link")):
            ref_id = text(l.find(nstag("ref-id")))
            title = text(l.find(nstag("title")))
            url = text(l.find(nstag("url")))
            out.append((
                "link",
                ref_id[:128] if ref_id else None,
                None, None, None,
                title[:512] if title else None,
                url
            ))

    # attachments/attachment => ref-id, title, url
    atts = ref_list_elem.find(nstag("attachments"))
    if atts is not None:
        for a in atts.findall(nstag("attachment")):
            ref_id = text(a.find(nstag("ref-id")))
            title = text(a.find(nstag("title")))
            url = text(a.find(nstag("url")))
            out.append((
                "attachment",
                ref_id[:128] if ref_id else None,
                None, None, None,
                title[:512] if title else None,
                url
            ))

    return out

def get_or_create_reference_pk(cur, cache: Dict[Tuple, int], ref_tuple: Tuple) -> int:
    if ref_tuple in cache:
        return cache[ref_tuple]

    (ref_type, ref_id, pubmed_id, citation, isbn, title, url) = ref_tuple

    # Fast lookup by ref_type, ref_id, pubmed_id
    cur.execute(SQL_REF_SELECT, (ref_type, ref_id, pubmed_id))
    row = cur.fetchone()
    if row:
        pk = int(row[0])
        cache[ref_tuple] = pk
        return pk

    cur.execute(SQL_REF_INSERT, (ref_type, ref_id, pubmed_id, citation, isbn, title, url))
    pk = int(cur.lastrowid)
    cache[ref_tuple] = pk
    return pk

# ---------------------------
# MAIN STAGE 4 RUNNER
# ---------------------------
print("Stage 4: counting total <drug> in XML (pre-pass)...")
total_drugs_xml = count_total_drugs_in_xml(XML_PATH)
print(f"Total drugs in XML: {total_drugs_xml:,}")

ns = detect_namespace(XML_PATH)
nstag = nstag_factory(ns)

conn_check = get_conn()
drug_pk_map = load_drug_pk_map(conn_check)

if not drug_pk_map:
    conn_check.close()
    raise RuntimeError("drug table is empty. Run Stage 1 first.")

if DO_INTERACTANT_REFERENCE:
    if not table_exists(conn_check, "interactant") or count_table(conn_check, "interactant") == 0:
        print("NOTICE: 'interactant' table is missing or empty. Disabling interactant_reference for this pass.")
        DO_INTERACTANT_REFERENCE = False

conn_check.close()

conn = get_conn()
cur = conn.cursor()

ref_cache: Dict[Tuple, int] = {}

ref_attempted = 0
ref_created = 0
drugref_attempted = 0
drugref_inserted = 0
intref_attempted = 0
intref_inserted = 0

missing_drug = 0
missing_interactant = 0
errors = 0

processed_drugs = 0
t0 = time.time()

pbar = tqdm(total=total_drugs_xml, unit="drug", dynamic_ncols=True, desc="Stage 4: References")

try:
    ctx = ET.iterparse(XML_PATH, events=("end",), huge_tree=True)
    for _, elem in ctx:
        if strip_ns(elem.tag) != "drug":
            continue

        primary_id = get_primary_drugbank_id(elem, nstag)
        if not primary_id:
            cleanup_lxml(elem)
            continue

        drug_pk = drug_pk_map.get(primary_id)
        if drug_pk is None:
            missing_drug += 1
            processed_drugs += 1
            pbar.update(1)
            cleanup_lxml(elem)
            continue

        try:
            # A) drug_reference
            genrefs = elem.find(nstag("general-references"))
            if genrefs is not None:
                refs = parse_reference_list(genrefs, nstag)
                for ref_tuple in refs:
                    ref_attempted += 1
                    reference_pk = get_or_create_reference_pk(cur, ref_cache, ref_tuple)

                    cur.execute(SQL_DRUG_REF_INSERT, (drug_pk, reference_pk))
                    drugref_attempted += 1
                    drugref_inserted += max(cur.rowcount, 0)

            # B) interactant_reference (if enabled and interactant table populated)
            if DO_INTERACTANT_REFERENCE:
                interactant_specs = [
                    ("target", "targets", "target"),
                    ("enzyme", "enzymes", "enzyme"),
                    ("carrier", "carriers", "carrier"),
                    ("transporter", "transporters", "transporter"),
                ]

                for kind, container_tag, item_tag in interactant_specs:
                    container = elem.find(nstag(container_tag))
                    if container is None:
                        continue

                    for item in container.findall(nstag(item_tag)):
                        interactant_id = text(item.find(nstag("id")))

                        cur.execute(SQL_INTERACTANT_FIND, (drug_pk, kind, interactant_id))
                        row = cur.fetchone()
                        if not row:
                            missing_interactant += 1
                            continue
                        interactant_pk = int(row[0])

                        refs_node = item.find(nstag("references"))
                        if refs_node is None:
                            continue

                        refs = parse_reference_list(refs_node, nstag)
                        for ref_tuple in refs:
                            ref_attempted += 1
                            reference_pk = get_or_create_reference_pk(cur, ref_cache, ref_tuple)

                            cur.execute(SQL_INTERACTANT_REF_INSERT, (interactant_pk, reference_pk))
                            intref_attempted += 1
                            intref_inserted += max(cur.rowcount, 0)

            processed_drugs += 1
            pbar.update(1)

            if processed_drugs % COMMIT_EVERY == 0:
                conn.commit()

        except Exception as e:
            errors += 1
            conn.rollback()
            print(f"[Stage4] ERROR drug_pk={drug_pk} ({primary_id}): {e}")

        finally:
            cleanup_lxml(elem)

    conn.commit()

finally:
    try: pbar.close()
    except Exception: pass
    cur.close()
    conn.close()

seconds = time.time() - t0

# Post-run Summary
conn_post = get_conn()
ref_rows = count_table(conn_post, "reference_item") if table_exists(conn_post, "reference_item") else -1
drugref_rows = count_table(conn_post, "drug_reference") if table_exists(conn_post, "drug_reference") else -1
intref_rows = count_table(conn_post, "interactant_reference") if (DO_INTERACTANT_REFERENCE and table_exists(conn_post, "interactant_reference")) else -1
conn_post.close()

headers = ["Metric", "Value"]
rows = [
    ["Total drugs in XML", f"{total_drugs_xml:,}"],
    ["Processed drugs", f"{processed_drugs:,}"],
    ["Missing drug (not in DB)", f"{missing_drug:,}"],
    ["Reference tuples seen", f"{ref_attempted:,}"],
    ["Drug_reference inserted", f"{drugref_inserted:,}"],
    ["Interactant_reference inserted", f"{intref_inserted:,}"],
    ["Missing interactant matches", f"{missing_interactant:,}"],
    ["Errors", f"{errors:,}"],
    ["Elapsed", f"{seconds:.1f}s"],
    ["DB rows: reference_item", "N/A" if ref_rows < 0 else f"{ref_rows:,}"],
    ["DB rows: drug_reference", "N/A" if drugref_rows < 0 else f"{drugref_rows:,}"],
    ["DB rows: interactant_reference", "N/A" if intref_rows < 0 else f"{intref_rows:,}"],
]

w = [len(h) for h in headers]
for r in rows:
    for i, v in enumerate(r):
        w[i] = max(w[i], len(str(v)))
def sep(ch="-"): return "+".join(ch * (x + 2) for x in w)
def line(r): return "|".join(" " + str(r[i]).ljust(w[i]) + " " for i in range(2))

print(sep("-"))
print(line(headers))
print(sep("="))
for r in rows:
    print(line(r))
    print(sep("-"))

Stage 4: counting total <drug> in XML (pre-pass)...
Total drugs in XML: 73,687
NOTICE: 'interactant' table is missing or empty. Disabling interactant_reference for this pass.


Stage 4: References:   0%|          | 0/73687 [00:00<?, ?drug/s]

--------------------------------+--------
 Metric                         | Value  
================================+========
 Total drugs in XML             | 73,687 
--------------------------------+--------
 Processed drugs                | 73,687 
--------------------------------+--------
 Missing drug (not in DB)       | 0      
--------------------------------+--------
 Reference tuples seen          | 25,607 
--------------------------------+--------
 Drug_reference inserted        | 0      
--------------------------------+--------
 Interactant_reference inserted | 0      
--------------------------------+--------
 Missing interactant matches    | 0      
--------------------------------+--------
 Errors                         | 0      
--------------------------------+--------
 Elapsed                        | 203.7s 
--------------------------------+--------
 DB rows: reference_item        | 23,879 
--------------------------------+--------
 DB rows: drug_reference        | 

<small>

### Stage 5 — Pathway joins and pathway members

This stage loads **pathway-related tables**:

- **`pathway`** *(independent; PK = `smpdb_id`)*  
- **`drug_pathway`** *(depends on `drug` + `pathway`)*  
- **`pathway_drug_member`** *(depends on `pathway`)*  
- **`pathway_enzyme_member`** *(depends on `pathway`)*  

---

#### Prerequisites
Before running Stage 5, you must have:

- **Stage 1:** `drug` loaded (so `drug_pk` exists)

---

#### What the Stage 5 cell does

**1) Counts total drugs first (pre-pass)**  
- Makes a streaming pass over the XML to count total `<drug>` elements.  
- Uses this total to show a **true progress bar**.

**2) Main insertion pass (with progress)**  
- Streams the XML again and processes each `<drug>`:
  - For each `<pathway>` inside `<pathways>`:
    - Upserts into **`pathway(smpdb_id, name, category)`**
    - Inserts the link into **`drug_pathway(drug_pk, smpdb_id)`**
    - Inserts pathway drug members into **`pathway_drug_member(smpdb_id, drugbank_id, name)`**
    - Inserts pathway enzymes into **`pathway_enzyme_member`** using UniProt IDs

---

#### Notes about `pathway_enzyme_member`
DrugBank XML provides pathway enzymes as a list of **`<uniprot-id>`** values.

The loader automatically detects whether your table uses:
- `uniprot_id` column (preferred), or
- `drugbank_id` column (fallback: stores the UniProt ID in that column)

---

#### Insert behavior
- `pathway`: **UPSERT** (`ON DUPLICATE KEY UPDATE`)
- `drug_pathway`: `INSERT IGNORE`
- `pathway_drug_member`: UPSERT (updates member `name`)
- `pathway_enzyme_member`: `INSERT IGNORE`

---

#### Output / reporting
The stage prints:
- Pre-pass drug count
- Progress bar during insertion
- Periodic status logs (counts and rate)
- Final summary table:
  - Attempted vs affected rows per table
  - Total rows in DB for each Stage 5 table

</small>


In [32]:
# =============================================================================
# STAGE 5 (Jupyter single cell) - HARDENED & CRASH-PROOF
# Builds:
#   - pathway               (smpdb_id PK)
#   - drug_pathway          (drug_pk + smpdb_id)
#   - pathway_drug_member   (smpdb_id + drugbank_id)
#   - pathway_enzyme_member (smpdb_id + drugbank_id/uniprot_id)
# =============================================================================

import time
from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Tuple

from lxml import etree as ET
from tqdm.notebook import tqdm
import mysql.connector

# ---------------------------
# CONFIGURATION
# ---------------------------
XML_PATH = r"db/drugbank.xml"

MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "niraj"
MYSQL_PASSWORD = "Niraj@123"
MYSQL_DB = "drugbank"

COMMIT_EVERY = 1000
SHOW_EVERY = 10000
# ---------------------------

def get_conn():
    return mysql.connector.connect(
        host=MYSQL_HOST, port=MYSQL_PORT, user=MYSQL_USER,
        password=MYSQL_PASSWORD, database=MYSQL_DB, autocommit=False
    )

def strip_ns(tag: str) -> str:
    return tag.split("}", 1)[1] if "}" in tag else tag

def detect_namespace(xml_path: str) -> Optional[str]:
    ctx = ET.iterparse(xml_path, events=("start",))
    _, root = next(ctx)
    if root.tag.startswith("{"):
        return root.tag.split("}")[0].strip("{")
    return None

def nstag_factory(ns: Optional[str]):
    def nstag(local: str) -> str:
        return f"{{{ns}}}{local}" if ns else local
    return nstag

def text(elem) -> Optional[str]:
    if elem is None or elem.text is None:
        return None
    t = elem.text.strip()
    return t if t else None

def cleanup_lxml(elem):
    elem.clear()
    while elem.getprevious() is not None:
        del elem.getparent()[0]

def bool_from_text(s: Optional[str]) -> bool:
    if s is None:
        return False
    v = s.strip().lower()
    return v in ("true", "1", "yes")

def get_primary_drugbank_id(drug_elem, nstag) -> Optional[str]:
    ids = drug_elem.findall(nstag("drugbank-id"))
    fallback = None
    primary = None
    for ide in ids:
        did = text(ide)
        if not did:
            continue
        if fallback is None:
            fallback = did
        if bool_from_text(ide.get("primary")):
            primary = did
            break
    return primary or fallback

def count_total_drugs_in_xml(xml_path: str) -> int:
    total = 0
    ctx = ET.iterparse(xml_path, events=("end",), huge_tree=True)
    for _, elem in ctx:
        if strip_ns(elem.tag) == "drug":
            total += 1
            cleanup_lxml(elem)
    return total

def load_drug_pk_map(conn) -> Dict[str, int]:
    cur = conn.cursor()
    cur.execute("SELECT primary_drugbank_id, drug_pk FROM drug")
    mp = {row[0]: int(row[1]) for row in cur.fetchall()}
    cur.close()
    return mp

def get_table_columns(conn, table: str) -> List[str]:
    cur = conn.cursor()
    cur.execute(
        "SELECT column_name FROM information_schema.columns "
        "WHERE table_schema=%s AND table_name=%s",
        (MYSQL_DB, table),
    )
    cols = [r[0] for r in cur.fetchall()]
    cur.close()
    return cols

def count_table(conn, table: str) -> int:
    cur = conn.cursor()
    cur.execute(f"SELECT COUNT(*) FROM `{table}`")
    n = int(cur.fetchone()[0])
    cur.close()
    return n

def print_table(rows: List[List[Any]], headers: List[str]) -> None:
    cols = len(headers)
    widths = [len(h) for h in headers]
    for r in rows:
        for i in range(cols):
            widths[i] = max(widths[i], len(str(r[i])))

    def sep(ch="-"):
        return "+".join(ch * (w + 2) for w in widths)

    def render(r):
        return "|".join(" " + str(r[i]).ljust(widths[i]) + " " for i in range(cols))

    print(sep("-"))
    print(render(headers))
    print(sep("="))
    for r in rows:
        print(render(r))
        print(sep("-"))

# ---------------------------
# PRE-PASS
# ---------------------------
print("Stage 5 pre-pass: counting total <drug> in XML...")
total_drugs_xml = count_total_drugs_in_xml(XML_PATH)
print(f"Total drugs in XML: {total_drugs_xml:,}")

# ---------------------------
# PREPARE DB & SCHEMA DETECTION
# ---------------------------
conn = get_conn()
cur = conn.cursor()

drug_pk_map = load_drug_pk_map(conn)
if not drug_pk_map:
    cur.close()
    conn.close()
    raise RuntimeError("drug table is empty. Run Stage 1 (drug) first.")

pem_cols = get_table_columns(conn, "pathway_enzyme_member")
pem_has_uniprot = "uniprot_id" in pem_cols
pem_has_drugbank = "drugbank_id" in pem_cols

SQL_PATHWAY_UPSERT = """
INSERT INTO pathway (smpdb_id, name, category)
VALUES (%s, %s, %s)
ON DUPLICATE KEY UPDATE
  name=VALUES(name),
  category=VALUES(category)
"""

SQL_DRUG_PATHWAY = "INSERT IGNORE INTO drug_pathway (drug_pk, smpdb_id) VALUES (%s, %s)"

SQL_PATHWAY_DRUG_MEMBER = """
INSERT INTO pathway_drug_member (smpdb_id, drugbank_id, name)
VALUES (%s, %s, %s)
ON DUPLICATE KEY UPDATE name=VALUES(name)
"""

if pem_has_uniprot:
    SQL_PATHWAY_ENZ_MEMBER = "INSERT IGNORE INTO pathway_enzyme_member (smpdb_id, uniprot_id) VALUES (%s, %s)"
elif pem_has_drugbank:
    SQL_PATHWAY_ENZ_MEMBER = "INSERT IGNORE INTO pathway_enzyme_member (smpdb_id, drugbank_id) VALUES (%s, %s)"
else:
    cur.close()
    conn.close()
    raise RuntimeError("pathway_enzyme_member table structure unrecognised.")

# ---------------------------
# MAIN PASS
# ---------------------------
attempted_pathway = affected_pathway = 0
attempted_drug_pathway = affected_drug_pathway = 0
attempted_pdm = affected_pdm = 0
attempted_pem = affected_pem = 0

processed_drugs = 0
missing_drug_in_db = 0

ns = detect_namespace(XML_PATH)
nstag = nstag_factory(ns)

t0 = time.time()
pbar = tqdm(total=total_drugs_xml, unit="drug", dynamic_ncols=True, desc="Stage 5: pathways")

try:
    ctx = ET.iterparse(XML_PATH, events=("end",), huge_tree=True)
    for _, elem in ctx:
        if strip_ns(elem.tag) != "drug":
            continue

        primary_id = get_primary_drugbank_id(elem, nstag)
        if not primary_id:
            processed_drugs += 1
            pbar.update(1)
            cleanup_lxml(elem)
            continue

        drug_pk = drug_pk_map.get(primary_id)
        if drug_pk is None:
            missing_drug_in_db += 1
            processed_drugs += 1
            pbar.update(1)
            cleanup_lxml(elem)
            continue

        try:
            pathways = elem.find(nstag("pathways"))
            if pathways is not None:
                for pw in pathways.findall(nstag("pathway")):
                    smpdb_id = text(pw.find(nstag("smpdb-id")))
                    if not smpdb_id:
                        continue

                    pw_name = text(pw.find(nstag("name")))
                    pw_cat = text(pw.find(nstag("category")))

                    # 1. Pathway Upsert (Safe Truncation)
                    cur.execute(SQL_PATHWAY_UPSERT, (
                        smpdb_id[:64],
                        pw_name[:512] if pw_name else None,
                        pw_cat[:256] if pw_cat else None
                    ))
                    attempted_pathway += 1
                    affected_pathway += max(cur.rowcount, 0)

                    # 2. Drug Pathway Join
                    cur.execute(SQL_DRUG_PATHWAY, (drug_pk, smpdb_id[:64]))
                    attempted_drug_pathway += 1
                    affected_drug_pathway += max(cur.rowcount, 0)

                    # 3. Pathway Drug Members
                    pw_drugs = pw.find(nstag("drugs"))
                    if pw_drugs is not None:
                        for pd in pw_drugs.findall(nstag("drug")):
                            did = text(pd.find(nstag("drugbank-id")))
                            if not did:
                                continue
                            dnm = text(pd.find(nstag("name")))
                            cur.execute(SQL_PATHWAY_DRUG_MEMBER, (
                                smpdb_id[:64],
                                did[:32],
                                dnm[:512] if dnm else None
                            ))
                            attempted_pdm += 1
                            affected_pdm += max(cur.rowcount, 0)

                    # 4. Pathway Enzyme Members
                    pw_enz = pw.find(nstag("enzymes"))
                    if pw_enz is not None:
                        for uid in pw_enz.findall(nstag("uniprot-id")):
                            u = text(uid)
                            if not u:
                                continue
                            cur.execute(SQL_PATHWAY_ENZ_MEMBER, (smpdb_id[:64], u[:32]))
                            attempted_pem += 1
                            affected_pem += max(cur.rowcount, 0)

            processed_drugs += 1
            pbar.update(1)

            if processed_drugs % COMMIT_EVERY == 0:
                conn.commit()

        except Exception as e:
            conn.rollback()
            print(f"[Stage5] ERROR for drug_pk={drug_pk} ({primary_id}): {e}")

        finally:
            cleanup_lxml(elem)

    conn.commit()

finally:
    try: pbar.close()
    except Exception: pass
    cur.close()
    conn.close()

seconds = time.time() - t0

# Final DB Counts
conn2 = get_conn()
rows_pathway = count_table(conn2, "pathway")
rows_drug_pathway = count_table(conn2, "drug_pathway")
rows_pdm = count_table(conn2, "pathway_drug_member")
rows_pem = count_table(conn2, "pathway_enzyme_member")
conn2.close()

print("\n=== Stage 5 Summary (Hardened) ===")
print_table(
    [
        ["pathway", f"{attempted_pathway:,}", f"{affected_pathway:,}", f"{rows_pathway:,}"],
        ["drug_pathway", f"{attempted_drug_pathway:,}", f"{affected_drug_pathway:,}", f"{rows_drug_pathway:,}"],
        ["pathway_drug_member", f"{attempted_pdm:,}", f"{affected_pdm:,}", f"{rows_pdm:,}"],
        ["pathway_enzyme_member", f"{attempted_pem:,}", f"{affected_pem:,}", f"{rows_pem:,}"],
    ],
    headers=["Table", "Attempted", "Affected(rowcount)", "Rows in DB"],
)

Stage 5 pre-pass: counting total <drug> in XML...
Total drugs in XML: 73,687


Stage 5: pathways:   0%|          | 0/73687 [00:00<?, ?drug/s]


=== Stage 5 Summary (Hardened) ===
-----------------------+-----------+--------------------+------------
 Table                 | Attempted | Affected(rowcount) | Rows in DB 
=======================+===========+====================+============
 pathway               | 3,780     | 877                | 877        
-----------------------+-----------+--------------------+------------
 drug_pathway          | 3,780     | 3,780              | 3,780      
-----------------------+-----------+--------------------+------------
 pathway_drug_member   | 0         | 0                  | 0          
-----------------------+-----------+--------------------+------------
 pathway_enzyme_member | 71,969    | 15,198             | 15,198     
-----------------------+-----------+--------------------+------------


<small>

### Stage 6 — Reaction children (`reaction_element`, `reaction_enzyme`)

**Goal:** Populate the reaction child tables after `reaction` exists.

**Tables inserted**
- `reaction_element` *(FK → `reaction.reaction_pk`)*
- `reaction_enzyme` *(FK → `reaction.reaction_pk`)*

---

#### Prerequisites
Before running Stage 6:
- **Stage 1** must be loaded: `drug(primary_drugbank_id → drug_pk)`
- **Stage 2** must be loaded: `reaction` rows exist (so `reaction_pk` exists)

---

#### How the Stage 6 code works

**1) Pre-pass (with progress bar)**
- Streams the XML once to count total `<drug>` elements
- Uses this total for an accurate progress bar in the main pass

**2) Main pass (with progress bar)**
- Streams the XML again and processes each `<drug>`
- For each drug:
  - Loads DB reactions for that drug:
    - `SELECT reaction_pk, sequence_text FROM reaction WHERE drug_pk=...`
  - Reads XML `<reactions><reaction>...</reaction></reactions>`
  - Maps each XML reaction to a DB `reaction_pk`:
    - tries matching by `sequence_text` first
    - falls back to positional mapping if needed
  - Inserts:
    - left/right sides into `reaction_element`
    - enzymes into `reaction_enzyme`

---

#### Notes
- Uses batching + commits for performance.
- Reports progress, insert counts, fallback usage, and final DB row counts.

</small>


In [33]:
# =============================================================================
# STAGE 6 (Jupyter single cell) - HARDENED & CRASH-PROOF
# Inserts:
#    - reaction_element  (FK -> reaction.reaction_pk)
#    - reaction_enzyme   (FK -> reaction.reaction_pk)
# =============================================================================

import time
from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Tuple

from lxml import etree as ET
from tqdm.notebook import tqdm
import mysql.connector

# ---------------------------
# CONFIGURATION
# ---------------------------
XML_PATH = r"db/drugbank.xml"

MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "niraj"
MYSQL_PASSWORD = "Niraj@123"
MYSQL_DB = "drugbank"

COMMIT_EVERY = 1000
SHOW_EVERY = 10000
# ---------------------------

def get_conn():
    return mysql.connector.connect(
        host=MYSQL_HOST, port=MYSQL_PORT, user=MYSQL_USER,
        password=MYSQL_PASSWORD, database=MYSQL_DB, autocommit=False
    )

def strip_ns(tag: str) -> str:
    return tag.split("}", 1)[1] if "}" in tag else tag

def detect_namespace(xml_path: str) -> Optional[str]:
    ctx = ET.iterparse(xml_path, events=("start",))
    _, root = next(ctx)
    if root.tag.startswith("{"):
        return root.tag.split("}")[0].strip("{")
    return None

def nstag_factory(ns: Optional[str]):
    def nstag(local: str) -> str:
        return f"{{{ns}}}{local}" if ns else local
    return nstag

def text(elem) -> Optional[str]:
    if elem is None or elem.text is None:
        return None
    t = elem.text.strip()
    return t if t else None

def cleanup_lxml(elem):
    elem.clear()
    while elem.getprevious() is not None:
        del elem.getparent()[0]

def bool_from_text(s: Optional[str]) -> bool:
    if s is None:
        return False
    v = s.strip().lower()
    return v in ("true", "1", "yes")

def get_primary_drugbank_id(drug_elem, nstag) -> Optional[str]:
    ids = drug_elem.findall(nstag("drugbank-id"))
    fallback = None
    primary = None
    for ide in ids:
        did = text(ide)
        if not did:
            continue
        if fallback is None:
            fallback = did
        if bool_from_text(ide.get("primary")):
            primary = did
            break
    return primary or fallback

def load_drug_pk_map(conn) -> Dict[str, int]:
    cur = conn.cursor()
    cur.execute("SELECT primary_drugbank_id, drug_pk FROM drug")
    mp = {row[0]: int(row[1]) for row in cur.fetchall()}
    cur.close()
    return mp

def count_table(conn, table: str) -> int:
    cur = conn.cursor()
    cur.execute(f"SELECT COUNT(*) FROM `{table}`")
    n = int(cur.fetchone()[0])
    cur.close()
    return n

def print_table(rows: List[List[Any]], headers: List[str]) -> None:
    cols = len(headers)
    widths = [len(h) for h in headers]
    for r in rows:
        for i in range(cols):
            widths[i] = max(widths[i], len(str(r[i])))

    def sep(ch="-"):
        return "+".join(ch * (w + 2) for w in widths)

    def render(r):
        return "|".join(" " + str(r[i]).ljust(widths[i]) + " " for i in range(cols))

    print(sep("-"))
    print(render(headers))
    print(sep("="))
    for r in rows:
        print(render(r))
        print(sep("-"))

def count_total_drugs_with_progress(xml_path: str) -> int:
    total = 0
    p = tqdm(total=None, unit="drug", dynamic_ncols=True, desc="Pre-pass: counting <drug>")
    try:
        ctx = ET.iterparse(xml_path, events=("end",), huge_tree=True)
        for _, elem in ctx:
            if strip_ns(elem.tag) == "drug":
                total += 1
                p.update(1)
                cleanup_lxml(elem)
    finally:
        try:
            p.close()
        except Exception:
            pass
    return total

# ---------------------------
# OPTIMIZED SQL STATEMENTS
# ---------------------------
SQL_DB_REACTIONS_FOR_DRUG = """
SELECT reaction_pk, sequence_text
FROM reaction
WHERE drug_pk=%s
ORDER BY reaction_pk
"""

SQL_REACTION_ELEMENT_UPSERT = """
INSERT INTO reaction_element (reaction_pk, side, drugbank_id, name)
VALUES (%s,%s,%s,%s)
ON DUPLICATE KEY UPDATE
  drugbank_id=VALUES(drugbank_id),
  name=VALUES(name)
"""

SQL_REACTION_ENZYME_INSERT = """
INSERT IGNORE INTO reaction_enzyme (reaction_pk, drugbank_id, name, uniprot_id)
VALUES (%s,%s,%s,%s)
"""

def norm_seq(s: Optional[str]) -> str:
    return (s or "").strip()

def safe_pk_str(s: Optional[str], default_val: str, max_len: int = 32) -> str:
    val = (s or "").strip()
    if not val:
        return default_val
    return val[:max_len]

# ---------------------------
# PRE-PASS
# ---------------------------
print("Stage 6 pre-pass starting...")
total_drugs_xml = count_total_drugs_with_progress(XML_PATH)
print(f"Total drugs in XML: {total_drugs_xml:,}")

# ---------------------------
# MAIN PASS
# ---------------------------
ns = detect_namespace(XML_PATH)
nstag = nstag_factory(ns)

conn = get_conn()
cur = conn.cursor()

drug_pk_map = load_drug_pk_map(conn)
if not drug_pk_map:
    cur.close()
    conn.close()
    raise RuntimeError("drug table is empty. Run Stage 1 first.")

attempted_el = inserted_el = 0
attempted_en = inserted_en = 0
processed_drugs = 0
missing_drug = 0
missing_db_reaction = 0
unmatched_by_seq_used_positional = 0
errors = 0

t0 = time.time()
pbar = tqdm(total=total_drugs_xml, unit="drug", dynamic_ncols=True, desc="Stage 6: reactions children")

try:
    ctx = ET.iterparse(XML_PATH, events=("end",), huge_tree=True)
    for _, drug_elem in ctx:
        if strip_ns(drug_elem.tag) != "drug":
            continue

        primary_id = get_primary_drugbank_id(drug_elem, nstag)
        drug_pk = drug_pk_map.get(primary_id) if primary_id else None
        if drug_pk is None:
            missing_drug += 1
            processed_drugs += 1
            pbar.update(1)
            cleanup_lxml(drug_elem)
            continue

        try:
            # Load DB reactions for this drug
            cur.execute(SQL_DB_REACTIONS_FOR_DRUG, (drug_pk,))
            db_reactions = [(int(rpk), norm_seq(seq)) for (rpk, seq) in cur.fetchall()]

            # Parse XML reactions
            reactions = drug_elem.find(nstag("reactions"))
            if reactions is not None:
                pointer = 0  # scanning pointer into db_reactions

                for rxn in reactions.findall(nstag("reaction")):
                    xml_seq = norm_seq(text(rxn.find(nstag("sequence"))))

                    # 1. Sequence match
                    match_idx = None
                    for j in range(pointer, len(db_reactions)):
                        if db_reactions[j][1] == xml_seq:
                            match_idx = j
                            break

                    # 2. Positional fallback
                    if match_idx is None:
                        if pointer < len(db_reactions):
                            match_idx = pointer
                            unmatched_by_seq_used_positional += 1
                        else:
                            missing_db_reaction += 1
                            continue

                    reaction_pk = db_reactions[match_idx][0]
                    pointer = match_idx + 1

                    # reaction_element: left-element + right-element
                    left = rxn.find(nstag("left-element"))
                    if left is not None:
                        lid = text(left.find(nstag("drugbank-id")))
                        lname = text(left.find(nstag("name")))
                        cur.execute(SQL_REACTION_ELEMENT_UPSERT, (
                            reaction_pk, 
                            "left", 
                            lid[:32] if lid else None, 
                            lname[:512] if lname else None
                        ))
                        attempted_el += 1
                        inserted_el += max(cur.rowcount, 0)

                    right = rxn.find(nstag("right-element"))
                    if right is not None:
                        rid = text(right.find(nstag("drugbank-id")))
                        rname = text(right.find(nstag("name")))
                        cur.execute(SQL_REACTION_ELEMENT_UPSERT, (
                            reaction_pk, 
                            "right", 
                            rid[:32] if rid else None, 
                            rname[:512] if rname else None
                        ))
                        attempted_el += 1
                        inserted_el += max(cur.rowcount, 0)

                    # reaction_enzyme
                    enzs = rxn.find(nstag("enzymes"))
                    if enzs is not None:
                        for enz in enzs.findall(nstag("enzyme")):
                            raw_eid = text(enz.find(nstag("drugbank-id")))
                            uname = text(enz.find(nstag("name")))
                            raw_uniprot = text(enz.find(nstag("uniprot-id")))

                            # Ignore if both IDs are completely absent
                            if not raw_eid and not raw_uniprot:
                                continue

                            # Assign safe non-null values for Primary Key requirements
                            eid = safe_pk_str(raw_eid, default_val="N/A", max_len=32)
                            uniprot = safe_pk_str(raw_uniprot, default_val="N/A", max_len=32)

                            cur.execute(SQL_REACTION_ENZYME_INSERT, (
                                reaction_pk, 
                                eid, 
                                uname[:512] if uname else None, 
                                uniprot
                            ))
                            attempted_en += 1
                            inserted_en += max(cur.rowcount, 0)

            processed_drugs += 1
            pbar.update(1)

            if processed_drugs % COMMIT_EVERY == 0:
                conn.commit()

        except Exception as e:
            errors += 1
            conn.rollback()
            print(f"[Stage6] ERROR drug_pk={drug_pk} ({primary_id}): {e}")

        finally:
            cleanup_lxml(drug_elem)

    conn.commit()

finally:
    try:
        pbar.close()
    except Exception:
        pass
    cur.close()
    conn.close()

elapsed = time.time() - t0

# Final DB counts
conn2 = get_conn()
rows_el = count_table(conn2, "reaction_element")
rows_en = count_table(conn2, "reaction_enzyme")
conn2.close()

print("\n=== Stage 6 Summary (Hardened) ===")
print_table(
    [
        ["reaction_element", f"{attempted_el:,}", f"{inserted_el:,}", f"{rows_el:,}"],
        ["reaction_enzyme", f"{attempted_en:,}", f"{inserted_en:,}", f"{rows_en:,}"],
    ],
    headers=["Table", "Attempted", "Affected(rowcount)", "Rows in DB"],
)

print("\nOther stats:")
print(f"- Drugs in XML (pre-pass):             {total_drugs_xml:,}")
print(f"- Drugs processed:                     {processed_drugs:,}")
print(f"- Missing drugs in DB (Stage 1 miss): {missing_drug:,}")
print(f"- Missing DB reaction match:          {missing_db_reaction:,}")
print(f"- Used positional reaction fallback:  {unmatched_by_seq_used_positional:,}")
print(f"- Errors:                              {errors:,}")
print(f"- Elapsed:                             {elapsed:.1f}s")

Stage 6 pre-pass starting...


Pre-pass: counting <drug>: 0drug [00:00, ?drug/s]

Total drugs in XML: 73,687


Stage 6: reactions children:   0%|          | 0/73687 [00:00<?, ?drug/s]


=== Stage 6 Summary (Hardened) ===
------------------+-----------+--------------------+------------
 Table            | Attempted | Affected(rowcount) | Rows in DB 
==================+===========+====================+============
 reaction_element | 8,092     | 8,092              | 8,092      
------------------+-----------+--------------------+------------
 reaction_enzyme  | 3,737     | 3,737              | 3,737      
------------------+-----------+--------------------+------------

Other stats:
- Drugs in XML (pre-pass):             73,687
- Drugs processed:                     73,687
- Missing drugs in DB (Stage 1 miss): 0
- Missing DB reaction match:          0
- Used positional reaction fallback:  0
- Errors:                              0
- Elapsed:                             91.5s


<small>

### Stage 7 — Interactant group (targets / enzymes / carriers / transporters)

This stage loads the **interactant dependency chain**:

- **`interactant`** *(FK → `drug`)*  
- **`interactant_action`** *(FK → `interactant`)*  
- **`polypeptide`** *(needed so `interactant_polypeptide` FK is valid)*  
- **`interactant_polypeptide`** *(FK → `interactant` + `polypeptide`)*  
- **`reference_item`** *(inserted as needed)*  
- **`interactant_reference`** *(FK → `interactant` + `reference_item`)*  

---

#### Prerequisites
Before running Stage 7, you must have:

- **Stage 1:** `drug` loaded (so `drug_pk` exists)

---

#### What the Stage 7 cell does

**1) Pre-pass: count total drugs (with progress bar)**  
- Streams the XML once to count `<drug>` elements  
- Uses this number for the main progress bar

**2) Main pass: insert interactants (with progress bar)**  
- Streams the XML again and processes each `<drug>`
- For each drug, it processes 4 interactant groups:
  - `targets/target`
  - `enzymes/enzyme`
  - `carriers/carrier`
  - `transporters/transporter`

For every interactant item it:
- **Creates or finds** a matching row in `interactant` using:
  - `drug_pk + kind + position + id + name + organism + known_action`
- Inserts actions into `interactant_action`
- Inserts polypeptides into `polypeptide` (UPSERT on `(polypeptide_id, source)`)
- Inserts links into `interactant_polypeptide`
- Inserts references:
  - creates/gets `reference_item` (deduped by composite key)
  - inserts into `interactant_reference`

---

#### Insert behavior / rerun safety
- `interactant`: **get-or-create** (find first, otherwise insert)
- `interactant_action`: `INSERT IGNORE`
- `polypeptide`: UPSERT (`ON DUPLICATE KEY UPDATE`)
- `interactant_polypeptide`: `INSERT IGNORE`
- `reference_item`: deduped using cache + DB lookup
- `interactant_reference`: `INSERT IGNORE`

---

#### Output / reporting
The stage prints:
- pre-pass drug count
- progress bar during processing
- periodic status logs (insert counts + speed)
- final summary including:
  - DB row counts for `interactant`, `interactant_action`, `polypeptide`,
    `interactant_polypeptide`, `reference_item`, and `interactant_reference`
  - missing drugs (if any)
  - runtime and error count

</small>


In [35]:
# =============================================================================
# STAGE 7 (Jupyter single cell) - HARDENED & CRASH-PROOF
# Inserts:
#   35) interactant              (FK -> drug.drug_pk)
#   36) interactant_action       (FK -> interactant.interactant_pk)
#   37) polypeptide              (PK -> (polypeptide_id, source))
#       interactant_polypeptide  (FK -> interactant + polypeptide)
#   38) reference_item + interactant_reference
# =============================================================================

import time
from dataclasses import dataclass
from typing import Optional, Dict, Any, Tuple, List

from lxml import etree as ET
from tqdm.notebook import tqdm
import mysql.connector

# ---------------------------
# CONFIGURATION
# ---------------------------
XML_PATH = r"db/drugbank.xml"

MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "niraj"
MYSQL_PASSWORD = "Niraj@123"
MYSQL_DB = "drugbank"

COMMIT_EVERY = 500
SHOW_EVERY = 5000
# ---------------------------

def get_conn():
    return mysql.connector.connect(
        host=MYSQL_HOST, port=MYSQL_PORT, user=MYSQL_USER,
        password=MYSQL_PASSWORD, database=MYSQL_DB, autocommit=False
    )

def strip_ns(tag: str) -> str:
    return tag.split("}", 1)[1] if "}" in tag else tag

def detect_namespace(xml_path: str) -> Optional[str]:
    ctx = ET.iterparse(xml_path, events=("start",))
    _, root = next(ctx)
    if root.tag.startswith("{"):
        return root.tag.split("}")[0].strip("{")
    return None

def nstag_factory(ns: Optional[str]):
    def nstag(local: str) -> str:
        return f"{{{ns}}}{local}" if ns else local
    return nstag

def text(elem) -> Optional[str]:
    if elem is None or elem.text is None:
        return None
    t = elem.text.strip()
    return t if t else None

def cleanup_lxml(elem):
    elem.clear()
    while elem.getprevious() is not None:
        del elem.getparent()[0]

def bool_from_text(s: Optional[str]) -> bool:
    if s is None:
        return False
    v = s.strip().lower()
    return v in ("true", "1", "yes")

def safe_str(s: Optional[str], max_len: int) -> Optional[str]:
    if not s:
        return None
    val = s.strip()
    return val[:max_len] if val else None

def get_primary_drugbank_id(drug_elem, nstag) -> Optional[str]:
    ids = drug_elem.findall(nstag("drugbank-id"))
    fallback = None
    primary = None
    for ide in ids:
        did = text(ide)
        if not did:
            continue
        if fallback is None:
            fallback = did
        if bool_from_text(ide.get("primary")):
            primary = did
            break
    return primary or fallback

def load_drug_pk_map(conn) -> Dict[str, int]:
    cur = conn.cursor()
    cur.execute("SELECT primary_drugbank_id, drug_pk FROM drug")
    mp = {row[0]: int(row[1]) for row in cur.fetchall()}
    cur.close()
    return mp

def count_table(conn, table: str) -> int:
    cur = conn.cursor()
    cur.execute(f"SELECT COUNT(*) FROM `{table}`")
    n = int(cur.fetchone()[0])
    cur.close()
    return n

def print_table(rows: List[List[Any]], headers: List[str]) -> None:
    cols = len(headers)
    widths = [len(h) for h in headers]
    for r in rows:
        for i in range(cols):
            widths[i] = max(widths[i], len(str(r[i])))

    def sep(ch="-"):
        return "+".join(ch * (w + 2) for w in widths)

    def render(r):
        return "|".join(" " + str(r[i]).ljust(widths[i]) + " " for i in range(cols))

    print(sep("-"))
    print(render(headers))
    print(sep("="))
    for r in rows:
        print(render(r))
        print(sep("-"))

def count_total_drugs_with_progress(xml_path: str) -> int:
    total = 0
    p = tqdm(total=None, unit="drug", dynamic_ncols=True, desc="Pre-pass: counting <drug>")
    try:
        ctx = ET.iterparse(xml_path, events=("end",), huge_tree=True)
        for _, elem in ctx:
            if strip_ns(elem.tag) == "drug":
                total += 1
                p.update(1)
                cleanup_lxml(elem)
    finally:
        try: p.close()
        except Exception: pass
    return total

# ---------------------------
# REFERENCE PARSING & FAST LOOKUP
# ---------------------------
SQL_REF_SELECT = """
SELECT reference_pk
FROM reference_item
WHERE ref_type <=> %s
  AND ref_id <=> %s
  AND pubmed_id <=> %s
LIMIT 1
"""

SQL_REF_INSERT = """
INSERT INTO reference_item
  (ref_type, ref_id, pubmed_id, citation, isbn, title, url)
VALUES (%s,%s,%s,%s,%s,%s,%s)
"""

def parse_reference_list(refs_node, nstag):
    out = []
    arts = refs_node.find(nstag("articles"))
    if arts is not None:
        for a in arts.findall(nstag("article")):
            out.append((
                "article",
                safe_str(text(a.find(nstag("ref-id"))), 128),
                safe_str(text(a.find(nstag("pubmed-id"))), 32),
                text(a.find(nstag("citation"))),
                None, None, None
            ))
    tbs = refs_node.find(nstag("textbooks"))
    if tbs is not None:
        for t in tbs.findall(nstag("textbook")):
            out.append((
                "textbook",
                safe_str(text(t.find(nstag("ref-id"))), 128),
                None,
                text(t.find(nstag("citation"))),
                safe_str(text(t.find(nstag("isbn"))), 64),
                None, None
            ))
    links = refs_node.find(nstag("links"))
    if links is not None:
        for l in links.findall(nstag("link")):
            out.append((
                "link",
                safe_str(text(l.find(nstag("ref-id"))), 128),
                None, None, None,
                safe_str(text(l.find(nstag("title"))), 512),
                text(l.find(nstag("url")))
            ))
    atts = refs_node.find(nstag("attachments"))
    if atts is not None:
        for a in atts.findall(nstag("attachment")):
            out.append((
                "attachment",
                safe_str(text(a.find(nstag("ref-id"))), 128),
                None, None, None,
                safe_str(text(a.find(nstag("title"))), 512),
                text(a.find(nstag("url")))
            ))
    return out

def get_or_create_reference_pk(cur, cache: Dict[Tuple, int], ref_tuple: Tuple) -> int:
    if ref_tuple in cache:
        return cache[ref_tuple]

    (ref_type, ref_id, pubmed_id, citation, isbn, title, url) = ref_tuple

    cur.execute(SQL_REF_SELECT, (ref_type, ref_id, pubmed_id))
    row = cur.fetchone()
    if row:
        pk = int(row[0])
        cache[ref_tuple] = pk
        return pk

    cur.execute(SQL_REF_INSERT, ref_tuple)
    pk = int(cur.lastrowid)
    cache[ref_tuple] = pk
    return pk

# ---------------------------
# INTERACTANT & POLYPEPTIDE SQL
# ---------------------------
SQL_INTERACTANT_FIND = """
SELECT interactant_pk
FROM interactant
WHERE drug_pk=%s
  AND kind <=> %s
  AND interactant_id <=> %s
ORDER BY interactant_pk DESC
LIMIT 1
"""

SQL_INTERACTANT_INSERT = """
INSERT INTO interactant
  (drug_pk, kind, position, interactant_id, name, organism, known_action)
VALUES (%s,%s,%s,%s,%s,%s,%s)
"""

SQL_INTERACTANT_ACTION = "INSERT IGNORE INTO interactant_action (interactant_pk, action) VALUES (%s,%s)"
SQL_INTERACTANT_POLY = "INSERT IGNORE INTO interactant_polypeptide (interactant_pk, polypeptide_id, source) VALUES (%s,%s,%s)"
SQL_INTERACTANT_REF = "INSERT IGNORE INTO interactant_reference (interactant_pk, reference_pk) VALUES (%s,%s)"

SQL_POLYPEPTIDE_UPSERT = """
INSERT INTO polypeptide (
  polypeptide_id, source, name, general_function, specific_function, gene_name, locus,
  cellular_location, transmembrane_regions, signal_regions, theoretical_pi, molecular_weight,
  chromosome_location, organism_name, organism_ncbi_taxonomy_id,
  amino_acid_sequence, gene_sequence
)
VALUES (%s,%s,%s,%s,%s,%s,%s,
        %s,%s,%s,%s,%s,
        %s,%s,%s,
        %s,%s)
ON DUPLICATE KEY UPDATE
  name=VALUES(name),
  general_function=VALUES(general_function),
  specific_function=VALUES(specific_function),
  gene_name=VALUES(gene_name),
  locus=VALUES(locus),
  cellular_location=VALUES(cellular_location),
  transmembrane_regions=VALUES(transmembrane_regions),
  signal_regions=VALUES(signal_regions),
  theoretical_pi=VALUES(theoretical_pi),
  molecular_weight=VALUES(molecular_weight),
  chromosome_location=VALUES(chromosome_location),
  organism_name=VALUES(organism_name),
  organism_ncbi_taxonomy_id=VALUES(organism_ncbi_taxonomy_id),
  amino_acid_sequence=VALUES(amino_acid_sequence),
  gene_sequence=VALUES(gene_sequence)
"""

def get_or_create_interactant_pk(cur, drug_pk: int, kind: str, position: Optional[int],
                                interactant_id: Optional[str], name: Optional[str],
                                organism: Optional[str], known_action: Optional[str]) -> int:
    cur.execute(SQL_INTERACTANT_FIND, (drug_pk, kind, interactant_id))
    row = cur.fetchone()
    if row:
        return int(row[0])
    cur.execute(SQL_INTERACTANT_INSERT, (drug_pk, kind, position, interactant_id, name, organism, known_action))
    return int(cur.lastrowid)

def safe_int(s: Optional[str]) -> Optional[int]:
    if s is None:
        return None
    try:
        return int(s)
    except Exception:
        return None

# ---------------------------
# RUN STAGE 7
# ---------------------------
print("Stage 7 pre-pass starting...")
total_drugs_xml = count_total_drugs_with_progress(XML_PATH)
print(f"Total drugs in XML: {total_drugs_xml:,}")

ns = detect_namespace(XML_PATH)
nstag = nstag_factory(ns)

conn = get_conn()
cur = conn.cursor()

drug_pk_map = load_drug_pk_map(conn)
if not drug_pk_map:
    cur.close()
    conn.close()
    raise RuntimeError("drug table is empty. Run Stage 1 first.")

# Counters
attempt_i = ins_i = 0
attempt_a = ins_a = 0
attempt_poly = ins_poly = 0
attempt_link = ins_link = 0
attempt_ref = ins_ref = 0

processed_drugs = 0
missing_drug = 0
errors = 0

ref_cache: Dict[Tuple, int] = {}

t0 = time.time()
pbar = tqdm(total=total_drugs_xml, unit="drug", dynamic_ncols=True, desc="Stage 7: interactants")

try:
    ctx = ET.iterparse(XML_PATH, events=("end",), huge_tree=True)
    for _, drug_elem in ctx:
        if strip_ns(drug_elem.tag) != "drug":
            continue

        primary_id = get_primary_drugbank_id(drug_elem, nstag)
        drug_pk = drug_pk_map.get(primary_id) if primary_id else None
        if drug_pk is None:
            missing_drug += 1
            processed_drugs += 1
            pbar.update(1)
            cleanup_lxml(drug_elem)
            continue

        try:
            groups = [
                ("target", "targets", "target"),
                ("enzyme", "enzymes", "enzyme"),
                ("carrier", "carriers", "carrier"),
                ("transporter", "transporters", "transporter"),
            ]

            for kind, container_tag, item_tag in groups:
                container = drug_elem.find(nstag(container_tag))
                if container is None:
                    continue

                for item in container.findall(nstag(item_tag)):
                    pos_i = safe_int(item.get("position"))
                    interactant_id = safe_str(text(item.find(nstag("id"))), 128)
                    name = safe_str(text(item.find(nstag("name"))), 512)
                    organism = safe_str(text(item.find(nstag("organism"))), 512)
                    
                    raw_action = text(item.find(nstag("known-action")))
                    known_action = raw_action.lower() if raw_action and raw_action.lower() in ('yes', 'no', 'unknown') else 'unknown'

                    # 1. Interactant Get-or-Create
                    interactant_pk = get_or_create_interactant_pk(
                        cur, drug_pk, kind, pos_i, interactant_id, name, organism, known_action
                    )
                    attempt_i += 1
                    ins_i += 1

                    # 2. Actions
                    actions = item.find(nstag("actions"))
                    if actions is not None:
                        for act in actions.findall(nstag("action")):
                            v = text(act)
                            if not v:
                                continue
                            cur.execute(SQL_INTERACTANT_ACTION, (interactant_pk, safe_str(v, 128)))
                            attempt_a += 1
                            ins_a += max(cur.rowcount, 0)

                    # 3. References -> reference_item + interactant_reference
                    refs_node = item.find(nstag("references"))
                    if refs_node is not None:
                        for ref_tuple in parse_reference_list(refs_node, nstag):
                            reference_pk = get_or_create_reference_pk(cur, ref_cache, ref_tuple)
                            attempt_ref += 1
                            cur.execute(SQL_INTERACTANT_REF, (interactant_pk, reference_pk))
                            attempt_link += 1
                            ins_link += max(cur.rowcount, 0)

                    # 4. Polypeptides
                    for poly in item.findall(nstag("polypeptide")):
                        poly_id = safe_str(poly.get("id"), 128)
                        poly_source = safe_str(poly.get("source"), 64)
                        if not poly_id or not poly_source:
                            continue

                        org_elem = poly.find(nstag("organism"))
                        org_name = safe_str(text(org_elem), 512) if org_elem is not None else None
                        org_tax = safe_str(org_elem.get("ncbi-taxonomy-id"), 64) if org_elem is not None else None

                        aa_seq = text(poly.find(nstag("amino-acid-sequence")))
                        gene_seq = text(poly.find(nstag("gene-sequence")))

                        cur.execute(SQL_POLYPEPTIDE_UPSERT, (
                            poly_id, poly_source,
                            safe_str(text(poly.find(nstag("name"))), 512),
                            text(poly.find(nstag("general-function"))),
                            text(poly.find(nstag("specific-function"))),
                            safe_str(text(poly.find(nstag("gene-name"))), 256),
                            safe_str(text(poly.find(nstag("locus"))), 256),
                            safe_str(text(poly.find(nstag("cellular-location"))), 256),
                            text(poly.find(nstag("transmembrane-regions"))),
                            text(poly.find(nstag("signal-regions"))),
                            safe_str(text(poly.find(nstag("theoretical-pi"))), 64),
                            safe_str(text(poly.find(nstag("molecular-weight"))), 64),
                            safe_str(text(poly.find(nstag("chromosome-location"))), 128),
                            org_name, org_tax,
                            aa_seq, gene_seq
                        ))
                        attempt_poly += 1
                        ins_poly += max(cur.rowcount, 0)

                        cur.execute(SQL_INTERACTANT_POLY, (interactant_pk, poly_id, poly_source))
                        attempt_link += 1
                        ins_link += max(cur.rowcount, 0)

            processed_drugs += 1
            pbar.update(1)

            if processed_drugs % COMMIT_EVERY == 0:
                conn.commit()

        except Exception as e:
            errors += 1
            conn.rollback()
            print(f"[Stage7] ERROR drug_pk={drug_pk} ({primary_id}): {e}")

        finally:
            cleanup_lxml(drug_elem)

    conn.commit()

finally:
    try:
        pbar.close()
    except Exception:
        pass
    cur.close()
    conn.close()

elapsed = time.time() - t0

# DB counts
conn2 = get_conn()
rows_interactant = count_table(conn2, "interactant")
rows_action = count_table(conn2, "interactant_action")
rows_poly = count_table(conn2, "polypeptide")
rows_link_poly = count_table(conn2, "interactant_polypeptide")
rows_refitem = count_table(conn2, "reference_item")
rows_intref = count_table(conn2, "interactant_reference")
conn2.close()

print("\n=== Stage 7 Summary (Hardened) ===")
print_table(
    [
        ["interactant", f"{attempt_i:,}", f"{rows_interactant:,}"],
        ["interactant_action", f"{attempt_a:,}", f"{rows_action:,}"],
        ["polypeptide", f"{attempt_poly:,}", f"{rows_poly:,}"],
        ["interactant_polypeptide", f"{attempt_link:,}", f"{rows_link_poly:,}"],
        ["reference_item (total)", "-", f"{rows_refitem:,}"],
        ["interactant_reference", f"{attempt_link:,}", f"{rows_intref:,}"],
    ],
    headers=["Table", "Attempted", "Rows in DB"],
)

print("\nOther stats:")
print(f"- Drugs in XML (pre-pass):             {total_drugs_xml:,}")
print(f"- Drugs processed:                     {processed_drugs:,}")
print(f"- Missing drugs in DB:                 {missing_drug:,}")
print(f"- Errors:                              {errors:,}")
print(f"- Elapsed:                             {elapsed:.1f}s")

Stage 7 pre-pass starting...


Pre-pass: counting <drug>: 0drug [00:00, ?drug/s]

Total drugs in XML: 73,687


Stage 7: interactants:   0%|          | 0/73687 [00:00<?, ?drug/s]


=== Stage 7 Summary (Hardened) ===
-------------------------+-----------+------------
 Table                   | Attempted | Rows in DB 
=========================+===========+============
 interactant             | 33,550    | 33,550     
-------------------------+-----------+------------
 interactant_action      | 25,388    | 25,388     
-------------------------+-----------+------------
 polypeptide             | 37,209    | 5,367      
-------------------------+-----------+------------
 interactant_polypeptide | 94,144    | 37,209     
-------------------------+-----------+------------
 reference_item (total)  | -         | 39,996     
-------------------------+-----------+------------
 interactant_reference   | 94,144    | 56,935     
-------------------------+-----------+------------

Other stats:
- Drugs in XML (pre-pass):             73,687
- Drugs processed:                     73,687
- Missing drugs in DB:                 0
- Errors:                              0
- Elapsed: 

<small>

### Stage 8 — Polypeptide children (`polypeptide_external_identifier`, `polypeptide_synonym`)

This stage loads the **child tables of `polypeptide`**:

- **`polypeptide_external_identifier`** *(FK → `polypeptide(polypeptide_id, source)`)*
- **`polypeptide_synonym`** *(FK → `polypeptide(polypeptide_id, source)`)*
  
---

#### Prerequisites
Before running Stage 8, you must have:

- **Stage 7:** `polypeptide` loaded (so `(polypeptide_id, source)` exists)

---

#### What the Stage 8 cell does

**1) Pre-pass: count total drugs (with progress bar)**  
- Streams the XML once to count `<drug>` elements  
- Uses this number for the main progress bar

**2) Main pass: insert polypeptide children (with progress bar)**  
- Streams the XML again and processes each `<drug>`
- Searches for polypeptides under the interactant groups:
  - `targets/target/polypeptide`
  - `enzymes/enzyme/polypeptide`
  - `carriers/carrier/polypeptide`
  - `transporters/transporter/polypeptide`

For each `<polypeptide id="..." source="...">` it:
- Verifies the polypeptide exists in the DB (FK safety)
- Inserts:
  - external identifiers into `polypeptide_external_identifier`
  - synonyms into `polypeptide_synonym`

---

#### Insert behavior / rerun safety
- Uses **`INSERT IGNORE`** for both tables  
  → safe to re-run without duplicating rows.

---

#### Output / reporting
The stage prints:
- pre-pass drug count
- progress bar during insertion
- periodic status logs (insert counts + speed)
- final summary table showing:
  - attempted statements
  - affected rows (`rowcount`)
  - total rows currently in DB for each Stage 8 table

It also reports:
- **missing polypeptide FK matches** (polypeptides found in XML but not present in `polypeptide` table)

</small>


In [37]:
# =============================================================================
# STAGE 8 (Jupyter single cell) - FAST & REAL-TIME STREAMING
# Inserts:
#   39) polypeptide_external_identifier (FK -> polypeptide(polypeptide_id, source))
#   40) polypeptide_synonym             (FK -> polypeptide(polypeptide_id, source))
# =============================================================================

import time, sys
from typing import Optional, Dict, Any, List, Tuple, Set

from lxml import etree as ET
from tqdm.notebook import tqdm
import mysql.connector

# ---------------------------
# CONFIGURATION
# ---------------------------
XML_PATH = r"db/drugbank.xml"

MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "niraj"
MYSQL_PASSWORD = "Niraj@123"
MYSQL_DB = "drugbank"

COMMIT_EVERY = 500
SHOW_EVERY = 1000
# ---------------------------

def get_conn():
    return mysql.connector.connect(
        host=MYSQL_HOST, port=MYSQL_PORT, user=MYSQL_USER,
        password=MYSQL_PASSWORD, database=MYSQL_DB, autocommit=False
    )

def strip_ns(tag: str) -> str:
    return tag.split("}", 1)[1] if "}" in tag else tag

def detect_namespace(xml_path: str) -> Optional[str]:
    ctx = ET.iterparse(xml_path, events=("start",))
    _, root = next(ctx)
    if root.tag.startswith("{"):
        return root.tag.split("}")[0].strip("{")
    return None

def nstag_factory(ns: Optional[str]):
    def nstag(local: str) -> str:
        return f"{{{ns}}}{local}" if ns else local
    return nstag

def text(elem) -> Optional[str]:
    if elem is None or elem.text is None:
        return None
    t = elem.text.strip()
    return t if t else None

def cleanup_lxml(elem):
    elem.clear()
    while elem.getprevious() is not None:
        del elem.getparent()[0]

def safe_str(s: Optional[str], max_len: int) -> Optional[str]:
    if not s:
        return None
    val = s.strip()
    return val[:max_len] if val else None

def count_table(conn, table: str) -> int:
    cur = conn.cursor()
    cur.execute(f"SELECT COUNT(*) FROM `{table}`")
    n = int(cur.fetchone()[0])
    cur.close()
    return n

def load_polypeptide_pk_set(conn) -> Set[Tuple[str, str]]:
    print("[INFO] Fetching polypeptide keys from database...", flush=True)
    cur = conn.cursor()
    cur.execute("SELECT polypeptide_id, source FROM polypeptide")
    s = {(row[0], row[1]) for row in cur.fetchall()}
    cur.close()
    print(f"[SUCCESS] Loaded {len(s):,} parent polypeptide keys.", flush=True)
    return s

SQL_POLY_EXT_ID = """
INSERT IGNORE INTO polypeptide_external_identifier (polypeptide_id, source, resource, identifier)
VALUES (%s,%s,%s,%s)
"""

SQL_POLY_SYN = """
INSERT IGNORE INTO polypeptide_synonym (polypeptide_id, source, synonym)
VALUES (%s,%s,%s)
"""

# ---------------------------
# MAIN EXECUTION
# ---------------------------
print("[INFO] Connecting to database...", flush=True)
conn = get_conn()
cur = conn.cursor()

poly_keys = load_polypeptide_pk_set(conn)
if not poly_keys:
    cur.close()
    conn.close()
    raise RuntimeError("polypeptide table is empty. Please run Stage 7 first.")

ns = detect_namespace(XML_PATH)
nstag = nstag_factory(ns)

attempt_ext = ins_ext = 0
attempt_syn = ins_syn = 0
processed_drugs = 0
missing_poly_fk = 0
errors = 0

t0 = time.time()
print("[INFO] Beginning XML processing for Stage 8...", flush=True)

# Using a single-pass streaming parser with tqdm progress feedback
pbar = tqdm(unit="drug", dynamic_ncols=True, desc="Stage 8: Polypeptide Children")

try:
    ctx = ET.iterparse(XML_PATH, events=("end",), huge_tree=True)
    for _, drug_elem in ctx:
        if strip_ns(drug_elem.tag) != "drug":
            continue

        try:
            groups = [
                ("targets", "target"),
                ("enzymes", "enzyme"),
                ("carriers", "carrier"),
                ("transporters", "transporter"),
            ]

            for container_tag, item_tag in groups:
                container = drug_elem.find(nstag(container_tag))
                if container is None:
                    continue

                for item in container.findall(nstag(item_tag)):
                    for poly in item.findall(nstag("polypeptide")):
                        poly_id = safe_str(poly.get("id"), 128)
                        poly_source = safe_str(poly.get("source"), 64)
                        if not poly_id or not poly_source:
                            continue

                        if (poly_id, poly_source) not in poly_keys:
                            missing_poly_fk += 1
                            continue

                        # External Identifiers
                        ex_ids = poly.find(nstag("external-identifiers"))
                        if ex_ids is not None:
                            for exi in ex_ids.findall(nstag("external-identifier")):
                                resource = safe_str(text(exi.find(nstag("resource"))), 256)
                                identifier = safe_str(text(exi.find(nstag("identifier"))), 256)
                                if not resource or not identifier:
                                    continue
                                cur.execute(SQL_POLY_EXT_ID, (poly_id, poly_source, resource, identifier))
                                attempt_ext += 1
                                ins_ext += max(cur.rowcount, 0)

                        # Synonyms
                        syns = poly.find(nstag("synonyms"))
                        if syns is not None:
                            for s in syns.findall(nstag("synonym")):
                                sv = safe_str(text(s), 512)
                                if not sv:
                                    continue
                                cur.execute(SQL_POLY_SYN, (poly_id, poly_source, sv))
                                attempt_syn += 1
                                ins_syn += max(cur.rowcount, 0)

            processed_drugs += 1
            pbar.update(1)

            if processed_drugs % COMMIT_EVERY == 0:
                conn.commit()

            if SHOW_EVERY and processed_drugs % SHOW_EVERY == 0:
                elapsed = time.time() - t0
                rate = processed_drugs / elapsed if elapsed > 0 else 0.0
                print(f"[Stage8] drugs={processed_drugs:,} ext_ins={ins_ext:,} syn_ins={ins_syn:,} rate={rate:.1f}/sec", flush=True)

        except Exception as e:
            errors += 1
            conn.rollback()
            print(f"[Stage8] ERROR at drug {processed_drugs}: {e}", flush=True)

        finally:
            cleanup_lxml(drug_elem)

    conn.commit()

finally:
    try: pbar.close()
    except Exception: pass
    cur.close()
    conn.close()

elapsed = time.time() - t0

# Final Verification
conn2 = get_conn()
rows_ext = count_table(conn2, "polypeptide_external_identifier")
rows_syn = count_table(conn2, "polypeptide_synonym")
rows_poly = count_table(conn2, "polypeptide")
conn2.close()

print("\n=== Stage 8 Final Summary ===", flush=True)
print(f"  • Processed Drugs: {processed_drugs:,}")
print(f"  • External IDs Inserted: {ins_ext:,} (DB Total: {rows_ext:,})")
print(f"  • Synonyms Inserted: {ins_syn:,} (DB Total: {rows_syn:,})")
print(f"  • Total Time: {elapsed:.1f}s")

[INFO] Connecting to database...
[INFO] Fetching polypeptide keys from database...
[SUCCESS] Loaded 5,367 parent polypeptide keys.
[INFO] Beginning XML processing for Stage 8...


Stage 8: Polypeptide Children: 0drug [00:00, ?drug/s]

[Stage8] drugs=1,000 ext_ins=3,167 syn_ins=2,434 rate=729.9/sec
[Stage8] drugs=2,000 ext_ins=3,290 syn_ins=2,507 rate=1375.3/sec
[Stage8] drugs=3,000 ext_ins=3,560 syn_ins=2,706 rate=1874.0/sec
[Stage8] drugs=4,000 ext_ins=3,863 syn_ins=2,901 rate=2331.1/sec
[Stage8] drugs=5,000 ext_ins=4,293 syn_ins=3,233 rate=2622.5/sec
[Stage8] drugs=6,000 ext_ins=4,535 syn_ins=3,420 rate=3018.8/sec
[Stage8] drugs=7,000 ext_ins=4,757 syn_ins=3,648 rate=3390.6/sec
[Stage8] drugs=8,000 ext_ins=5,180 syn_ins=3,972 rate=3621.3/sec
[Stage8] drugs=9,000 ext_ins=5,509 syn_ins=4,219 rate=3785.9/sec
[Stage8] drugs=10,000 ext_ins=6,482 syn_ins=4,980 rate=3732.3/sec
[Stage8] drugs=11,000 ext_ins=6,767 syn_ins=5,156 rate=3945.3/sec
[Stage8] drugs=12,000 ext_ins=6,968 syn_ins=5,311 rate=4141.2/sec
[Stage8] drugs=13,000 ext_ins=7,485 syn_ins=5,686 rate=3921.3/sec
[Stage8] drugs=14,000 ext_ins=8,804 syn_ins=6,727 rate=3764.4/sec
[Stage8] drugs=15,000 ext_ins=9,407 syn_ins=7,197 rate=3736.1/sec
[Stage8] drugs=16,00

<small>

### Stage 9 — Optional raw storage (`drug_raw`)

This stage stores the **entire raw `<drug>` XML record** into the database:

- **`drug_raw`** *(FK → `drug.drug_pk`)*  
  This should be the **last step per drug**, after all structured tables are loaded.

---

#### Prerequisites
Before running Stage 9, you must have:

- **Stage 1:** `drug` loaded (so `drug_pk` exists)

---

#### What the Stage 9 cell does

**1) Pre-pass: count total drugs (with progress bar)**  
- Streams the XML once to count `<drug>` elements  
- Uses that total for the main progress bar

**2) Main pass: insert/update `drug_raw`**  
- Streams the XML again and for each `<drug>`:
  - resolves `drug_pk` using `primary_drugbank_id`
  - converts the XML subtree into a **JSON-serializable dict** (tag/attrs/text/children)
  - writes it into `drug_raw.raw_json` using:
    - `INSERT ... ON DUPLICATE KEY UPDATE`  
      (safe to re-run; it updates the stored raw JSON)

---

#### MariaDB / MySQL notes
- The loader stores JSON as **TEXT/LONGTEXT** (no `CAST(... AS JSON)`).
- Recommended schema:
  - `ALTER TABLE drug_raw MODIFY raw_json LONGTEXT NOT NULL;`

If you hit `max_allowed_packet` issues (very large raw records), enable optional compression:
- set `COMPRESS_RAW = True`
- data will be stored as a string with prefix: `zlib+b64:...`

---

#### Output / reporting
The stage prints:
- pre-pass drug count
- progress bar during insertion
- periodic status logs (writes + speed)
- final summary table:
  - attempted writes
  - affected rows (`rowcount`)
  - total rows in `drug_raw`
  - runtime and error stats

</small>


In [38]:
# =============================================================================
# STAGE 9 (Jupyter single cell) - HARDENED & CRASH-PROOF
# Inserts:
#    - drug_raw  (drug_pk PK, raw_json LONGTEXT)
# =============================================================================

import time, base64, zlib
from lxml import etree as ET
from tqdm.notebook import tqdm
import mysql.connector

# ---------------------------
# CONFIGURATION
# ---------------------------
XML_PATH = r"db/drugbank.xml"

MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "niraj"
MYSQL_PASSWORD = "Niraj@123"
MYSQL_DB = "drugbank"

COMMIT_EVERY = 200
SHOW_EVERY = 2000

# 1GB Session max_allowed_packet setting
SESSION_MAX_PACKET = 1073741824

COMPRESS_RAW = True
# ---------------------------

SQL_DRUG_RAW_UPSERT = """
INSERT INTO drug_raw (drug_pk, raw_json)
VALUES (%s, %s)
ON DUPLICATE KEY UPDATE raw_json=VALUES(raw_json)
"""

def open_conn():
    conn = mysql.connector.connect(
        host=MYSQL_HOST, port=MYSQL_PORT, user=MYSQL_USER,
        password=MYSQL_PASSWORD, database=MYSQL_DB, autocommit=False
    )
    cur = conn.cursor()
    try:
        cur.execute(f"SET SESSION max_allowed_packet={SESSION_MAX_PACKET}")
    except Exception:
        pass
    return conn, cur

def strip_ns(tag: str) -> str:
    return tag.split("}", 1)[1] if "}" in tag else tag

def cleanup_lxml(elem):
    elem.clear()
    while elem.getprevious() is not None:
        del elem.getparent()[0]

def detect_namespace(xml_path: str):
    ctx = ET.iterparse(xml_path, events=("start",))
    _, root = next(ctx)
    if root.tag.startswith("{"):
        return root.tag.split("}")[0].strip("{")
    return None

def nstag_factory(ns):
    def nstag(local: str) -> str:
        return f"{{{ns}}}{local}" if ns else local
    return nstag

def text(elem):
    if elem is None or elem.text is None:
        return None
    t = elem.text.strip()
    return t if t else None

def pick_primary_id(drug_elem, nstag):
    ids = drug_elem.findall(nstag("drugbank-id"))
    first = None
    for ide in ids:
        did = text(ide)
        if not did:
            continue
        if first is None:
            first = did
        prim = ide.get("primary")
        if prim is not None:
            v = prim.strip().lower()
            if v not in ("false", "0", "no", "n", ""):
                return did
    return first

def load_drug_pk_map(cur):
    cur.execute("SELECT primary_drugbank_id, drug_pk FROM drug")
    return {row[0]: int(row[1]) for row in cur.fetchall()}

def count_total_drugs_with_progress(xml_path: str) -> int:
    total = 0
    p = tqdm(total=None, unit="drug", dynamic_ncols=True, desc="Pre-pass: counting <drug>")
    try:
        ctx = ET.iterparse(xml_path, events=("end",), huge_tree=True)
        for _, e in ctx:
            if strip_ns(e.tag) == "drug":
                total += 1
                p.update(1)
                cleanup_lxml(e)
    finally:
        try: p.close()
        except Exception: pass
    return total

def encode_raw(drug_elem) -> str:
    raw_bytes = ET.tostring(drug_elem, encoding="utf-8", with_tail=False)
    if not COMPRESS_RAW:
        return raw_bytes.decode("utf-8", errors="replace")
    comp = zlib.compress(raw_bytes, level=6)
    return "zlib+b64:" + base64.b64encode(comp).decode("ascii")

def count_table(cur, table: str) -> int:
    cur.execute(f"SELECT COUNT(*) FROM `{table}`")
    return int(cur.fetchone()[0])

def reconnect(conn, cur):
    try: cur.close()
    except Exception: pass
    try: conn.close()
    except Exception: pass
    return open_conn()

# ---------------------------
# RUN STAGE 9
# ---------------------------
print("Stage 9 pre-pass starting...")
total_drugs_xml = count_total_drugs_with_progress(XML_PATH)
print(f"Total drugs in XML: {total_drugs_xml:,}")

ns = detect_namespace(XML_PATH)
nstag = nstag_factory(ns)

conn, cur = open_conn()

drug_pk_map = load_drug_pk_map(cur)
if not drug_pk_map:
    raise RuntimeError("drug table is empty. Run Stage 1 first.")

attempted = 0
affected = 0
processed = 0
missing_drug = 0
too_large = 0
errors = 0

t0 = time.time()
pbar = tqdm(total=total_drugs_xml, unit="drug", dynamic_ncols=True, desc="Stage 9: drug_raw")

try:
    ctx = ET.iterparse(XML_PATH, events=("end",), huge_tree=True)
    for _, drug_elem in ctx:
        if strip_ns(drug_elem.tag) != "drug":
            continue

        primary_id = pick_primary_id(drug_elem, nstag)
        drug_pk = drug_pk_map.get(primary_id) if primary_id else None

        if drug_pk is None:
            missing_drug += 1
            processed += 1
            pbar.update(1)
            cleanup_lxml(drug_elem)
            continue

        # Retry loop for socket resets / connection drops
        success = False
        for attempt in range(2):
            try:
                payload = encode_raw(drug_elem)
                cur.execute(SQL_DRUG_RAW_UPSERT, (drug_pk, payload))
                attempted += 1
                affected += max(cur.rowcount, 0)
                success = True
                break

            except mysql.connector.Error as e:
                err = getattr(e, "errno", None)

                # Packet too large (Error 1153)
                if err == 1153:
                    too_large += 1
                    try: conn.rollback()
                    except Exception: pass
                    print(f"[Stage9] SKIP drug_pk={drug_pk} (packet too large).")
                    break

                # Connection reset / lost (Errors 2006, 2013, 2055) -> Retry once
                elif err in (2006, 2013, 2055):
                    print(f"[Stage9] Connection lost (errno={err}). Reconnecting and retrying drug_pk={drug_pk}...")
                    conn, cur = reconnect(conn, cur)
                else:
                    try: conn.rollback()
                    except Exception: pass
                    print(f"[Stage9] ERROR drug_pk={drug_pk} errno={err}: {e}")
                    break

        if not success and err not in (1153,):
            errors += 1

        processed += 1
        pbar.update(1)

        if processed % COMMIT_EVERY == 0:
            try:
                conn.commit()
            except mysql.connector.Error:
                conn, cur = reconnect(conn, cur)

        if SHOW_EVERY and processed % SHOW_EVERY == 0:
            elapsed = time.time() - t0
            rate = processed / elapsed if elapsed > 0 else 0.0
            print(f"[Stage9] drugs={processed:,}/{total_drugs_xml:,} writes={attempted:,} affected={affected:,} "
                  f"missing_drug={missing_drug:,} too_large={too_large:,} errors={errors:,} rate={rate:.2f}/sec")

        cleanup_lxml(drug_elem)

    try:
        conn.commit()
    except Exception:
        pass

finally:
    try: pbar.close()
    except Exception: pass
    try: cur.close()
    except Exception: pass
    try: conn.close()
    except Exception: pass

# Final summary
conn3, cur3 = open_conn()
rows_raw = count_table(cur3, "drug_raw")
cur3.close()
conn3.close()

elapsed = time.time() - t0
print("\n=== Stage 9 Summary (Hardened) ===")
print(f"Processed: {processed:,}/{total_drugs_xml:,}")
print(f"Writes attempted: {attempted:,} | affected(rowcount): {affected:,}")
print(f"Rows in drug_raw: {rows_raw:,}")
print(f"Missing drug: {missing_drug:,} | Too large skipped: {too_large:,} | Errors: {errors:,}")
print(f"Elapsed: {elapsed:.1f}s | COMPRESS_RAW={COMPRESS_RAW}")

Stage 9 pre-pass starting...


Pre-pass: counting <drug>: 0drug [00:00, ?drug/s]

Total drugs in XML: 73,687


Stage 9: drug_raw:   0%|          | 0/73687 [00:00<?, ?drug/s]

[Stage9] drugs=2,000/73,687 writes=2,000 affected=300 missing_drug=0 too_large=0 errors=0 rate=1076.59/sec
[Stage9] drugs=4,000/73,687 writes=4,000 affected=333 missing_drug=0 too_large=0 errors=0 rate=1620.99/sec
[Stage9] drugs=6,000/73,687 writes=6,000 affected=356 missing_drug=0 too_large=0 errors=0 rate=2033.33/sec
[Stage9] drugs=8,000/73,687 writes=8,000 affected=400 missing_drug=0 too_large=0 errors=0 rate=2397.08/sec
[Stage9] drugs=10,000/73,687 writes=10,000 affected=466 missing_drug=0 too_large=0 errors=0 rate=2597.10/sec
[Stage9] drugs=12,000/73,687 writes=12,000 affected=507 missing_drug=0 too_large=0 errors=0 rate=2875.52/sec
[Stage9] drugs=14,000/73,687 writes=14,000 affected=562 missing_drug=0 too_large=0 errors=0 rate=2919.58/sec
[Stage9] drugs=16,000/73,687 writes=16,000 affected=608 missing_drug=0 too_large=0 errors=0 rate=2966.54/sec
[Stage9] drugs=18,000/73,687 writes=18,000 affected=777 missing_drug=0 too_large=0 errors=0 rate=2070.82/sec
[Stage9] drugs=20,000/73,68

<small>

### Stage 9 — Optional raw storage (`drug_raw`)

This stage stores the **entire raw `<drug>` XML record** into the database:

- **`drug_raw`** *(FK → `drug.drug_pk`)*  
  This should be the **last step per drug**, after all structured tables are loaded.

---

#### Prerequisites
Before running Stage 9, you must have:

- **Stage 1:** `drug` loaded (so `drug_pk` exists)

---

#### What the Stage 9 cell does

**1) Pre-pass: count total drugs (with progress bar)**  
- Streams the XML once to count `<drug>` elements  
- Uses that total for the main progress bar

**2) Main pass: insert/update `drug_raw`**  
- Streams the XML again and for each `<drug>`:
  - resolves `drug_pk` using `primary_drugbank_id`
  - converts the XML subtree into a **JSON-serializable dict** (tag/attrs/text/children)
  - writes it into `drug_raw.raw_json` using:
    - `INSERT ... ON DUPLICATE KEY UPDATE`  
      (safe to re-run; it updates the stored raw JSON)

---

#### MariaDB / MySQL notes
- The loader stores JSON as **TEXT/LONGTEXT** (no `CAST(... AS JSON)`).
- Recommended schema:
  - `ALTER TABLE drug_raw MODIFY raw_json LONGTEXT NOT NULL;`

If you hit `max_allowed_packet` issues (very large raw records), enable optional compression:
- set `COMPRESS_RAW = True`
- data will be stored as a string with prefix: `zlib+b64:...`

---

# Count rows in all DrugBank tables (prints one summary table)

import mysql.connector

MYSQL_HOST="localhost"; MYSQL_PORT=3306; MYSQL_USER="niraj"; MYSQL_PASSWORD="Niraj@123"; MYSQL_DB="drugbank"

TABLES = [
    "drug","drugbank_id_map","drug_group","drug_classification","drug_classification_alternative_parent",
    "drug_classification_substituent","drug_synonym","drug_salt","product","packager","manufacturer",
    "drug_category","drug_affected_organism","drug_dosage","drug_ahfs_code","drug_pdb_entry",
    "drug_atc_code","drug_atc_level","drug_price","drug_patent","drug_food_interaction","drug_interaction",
    "drug_external_identifier","drug_external_link","drug_property","reaction","drug_salt_drugbank_id",
    "reference_item","drug_reference","interactant_reference","interactant","interactant_action",
    "interactant_polypeptide","polypeptide","polypeptide_external_identifier","polypeptide_synonym",
    "pathway","drug_pathway","pathway_drug_member","pathway_enzyme_member","reaction_element",
    "reaction_enzyme","drug_raw",
]

conn = mysql.connector.connect(host=MYSQL_HOST, port=MYSQL_PORT, user=MYSQL_USER,
                              password=MYSQL_PASSWORD, database=MYSQL_DB, autocommit=True)
cur = conn.cursor()

rows = []
for t in TABLES:
    cur.execute("SELECT COUNT(*) FROM information_schema.tables WHERE table_schema=%s AND table_name=%s", (MYSQL_DB, t))
    if int(cur.fetchone()[0]) == 0:
        rows.append((t, "N/A"))
        continue
    cur.execute(f"SELECT COUNT(*) FROM `{t}`")
    rows.append((t, f"{int(cur.fetchone()[0]):,}"))

cur.close(); conn.close()

w = max(len("Table"), *(len(t) for t,_ in rows))
print(f"{'Table'.ljust(w)}  Rows")
print("-"*(w+6))
for t,n in rows:
    print(f"{t.ljust(w)}  {n}")

#### Output / reporting
The stage prints:
- pre-pass drug count
- progress bar during insertion
- periodic status logs (writes + speed)
- final summary table:
  - attempted writes
  - affected rows (`rowcount`)
  - total rows in `drug_raw`
  - runtime and error stats

</small>


In [39]:
# Jupyter cell: Count rows in ALL tables (Dynamic & Fast Summary)
import mysql.connector

MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_USER = "niraj"
MYSQL_PASSWORD = "Niraj@123"
MYSQL_DB = "drugbank"

def get_conn():
    return mysql.connector.connect(
        host=MYSQL_HOST,
        port=MYSQL_PORT,
        user=MYSQL_USER,
        password=MYSQL_PASSWORD,
        database=MYSQL_DB,
        autocommit=True,
    )

def print_table(rows, headers):
    cols = len(headers)
    widths = [len(h) for h in headers]
    for r in rows:
        for i in range(cols):
            widths[i] = max(widths[i], len(str(r[i])))

    def sep(ch="-"):
        return "+".join(ch * (w + 2) for w in widths)

    def render(r):
        return "|".join(" " + str(r[i]).ljust(widths[i]) + " " for i in range(cols))

    print(sep("-"))
    print(render(headers))
    print(sep("="))
    for r in rows:
        print(render(r))
        print(sep("-"))

def audit_database_rows():
    conn = get_conn()
    cur = conn.cursor()

    try:
        # 1. Fetch all existing tables in the database dynamically
        cur.execute("""
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = %s 
            ORDER BY table_name ASC
        """, (MYSQL_DB,))
        existing_tables = [r[0] for r in cur.fetchall()]

        if not existing_tables:
            print(f"⚠️ Database '{MYSQL_DB}' contains 0 tables.")
            return

        rows = []
        total_rows = 0

        # 2. Get exact row counts for each table
        for table in existing_tables:
            try:
                cur.execute(f"SELECT COUNT(*) FROM `{table}`")
                count = int(cur.fetchone()[0])
                total_rows += count
                rows.append([table, f"{count:,}"])
            except mysql.connector.Error as err:
                rows.append([table, f"ERROR: {err.errno}"])

        # 3. Output summary table
        print_table(rows, headers=["Table", "Row count"])
        print("=" * 60)
        print(f"  Total Tables Counted : {len(existing_tables)}")
        print(f"  Total Rows In Database: {total_rows:,}")
        print("=" * 60)

    finally:
        cur.close()
        conn.close()

# Execute Audit
audit_database_rows()

----------------------------------------+-----------
 Table                                  | Row count 
========================================+===========
 drug                                   | 17,430    
----------------------------------------+-----------
 drug_affected_organism                 | 3,453     
----------------------------------------+-----------
 drug_ahfs_code                         | 0         
----------------------------------------+-----------
 drug_atc_code                          | 5,656     
----------------------------------------+-----------
 drug_atc_level                         | 22,624    
----------------------------------------+-----------
 drug_category                          | 101,512   
----------------------------------------+-----------
 drug_classification                    | 11,154    
----------------------------------------+-----------
 drug_classification_alternative_parent | 0         
----------------------------------------+-----

In [ ]:
### Cross Check

In [40]:
from lxml import etree as ET

XML_PATH = r"db/drugbank.xml"

ns = None
ctx = ET.iterparse(XML_PATH, events=("start",))
_, root = next(ctx)
ns = root.tag.split("}")[0].strip("{") if root.tag.startswith("{") else None

def nstag(x): 
    return f"{{{ns}}}{x}" if ns else x

count = 0
ctx2 = ET.iterparse(XML_PATH, events=("end",), huge_tree=True)
for _, e in ctx2:
    if e.tag == nstag("ahfs-codes"):
        count += 1
        e.clear()
        break
print("Found <ahfs-codes>:", count)


Found <ahfs-codes>: 1


In [41]:
count_ap = 0
count_sub = 0
ctx2 = ET.iterparse(XML_PATH, events=("end",), huge_tree=True)
for _, e in ctx2:
    if e.tag == nstag("alternative-parents"):
        count_ap += 1
        break
    if e.tag == nstag("substituents"):
        count_sub += 1
        break
print("Found <alternative-parents>:", count_ap)
print("Found <substituents>:", count_sub)


Found <alternative-parents>: 0
Found <substituents>: 0


In [42]:
count_pw_drugs = 0
ctx2 = ET.iterparse(XML_PATH, events=("end",), huge_tree=True)
for _, e in ctx2:
    if e.tag == nstag("drugs") and e.getparent() is not None and e.getparent().tag == nstag("pathway"):
        count_pw_drugs += 1
        break
print("Found <pathway><drugs>:", count_pw_drugs)


Found <pathway><drugs>: 1


In [43]:
from lxml import etree as ET

XML_PATH = r"db/drugbank.xml"

# detect namespace
ctx = ET.iterparse(XML_PATH, events=("start",))
_, root = next(ctx)
ns = root.tag.split("}")[0].strip("{") if root.tag.startswith("{") else None
def nstag(x): return f"{{{ns}}}{x}" if ns else x

targets = {
    "ahfs-codes": False,
    "alternative-parents": False,
    "substituents": False,
    "pathway_drugs_block": False,  # <pathway><drugs>
}

ctx2 = ET.iterparse(XML_PATH, events=("end",), huge_tree=True)
for _, e in ctx2:
    tag = e.tag
    if tag == nstag("ahfs-codes"):
        targets["ahfs-codes"] = True
    if tag == nstag("alternative-parents"):
        targets["alternative-parents"] = True
    if tag == nstag("substituents"):
        targets["substituents"] = True
    if tag == nstag("drugs") and e.getparent() is not None and e.getparent().tag == nstag("pathway"):
        targets["pathway_drugs_block"] = True

    if all(targets.values()):
        break

    e.clear()
    while e.getprevious() is not None:
        del e.getparent()[0]

targets

{'ahfs-codes': True,
 'alternative-parents': False,
 'substituents': False,
 'pathway_drugs_block': True}